# 0. Import Libraries

In [1]:
# Verify GPU is available
import tensorflow as tf
import torch

print("="*60)
print("GPU VERIFICATION")
print("="*60)

print("\nTensorFlow:")
print(f"  Version: {tf.__version__}")
print(f"  GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"  GPU Devices: {tf.config.list_physical_devices('GPU')}")

print("\nPyTorch:")
print(f"  CUDA Available: {torch.cuda.is_available()}")

print("\nNVIDIA GPU Info:")
!nvidia-smi --query-gpu=name,memory.total --format=csv

print("\n" + "="*60)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Import all necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import os
import joblib
from pathlib import Path
import pywt
import scipy.signal
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
from tensorflow import keras
from keras.models import load_model
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Reshape, GRU, LSTM, Input, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

os.environ['PYTHONHASHSEED'] = '42'
tf.random.set_seed(42)

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import KFold, StratifiedGroupKFold

print("✅ All libraries imported successfully!")


In [ ]:
AUDIO_PATH = '/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/'

MEL_1024_FEATURES_INPUT_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_features/mel_mfcc_features/features/"
MEL_2048_FEATURES_INPUT_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_features/mel_mfcc_features/features/"
MFCC_FEATURES_INPUT_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_features/mel_mfcc_features/features/"
TZANETAKIS_FEATURES_INPUT_PATH = "/kaggle/input/music-genre-classification-dataset/tzanetakis/tzanetakis/features/"
OUTPUT_FEATURES_PATH = "/kaggle/working/features/"

MEL_1024_MODEL_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/model/"
MEL_2048_MODEL_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/model/"
MFCC_MODEL_PATH ="/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/model/"
TZANETAKIS_MODEL_PATH = "/kaggle/input/music-genre-classification-dataset/tzanetakis/tzanetakis/model/"
TZANETAKIS_D_MODEL_PATH = "/kaggle/input/music-genre-classification-dataset/tzanetakis/tzanetakis/model/"

MEL_2048_D_MODEL_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/model/"

MEL_1024_HISTORY_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/history/"
MEL_2048_HISTORY_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/history/"

MEL_2048_D_HISTORY_PATH = "/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/history/"
MFCC_HISTORY_PATH ="/kaggle/input/music-genre-classification-dataset/mel_mfcc_toptimized/mel_mfcc_toptimized/history/"
TZANETAKIS_HISTORY_PATH = "/kaggle/input/music-genre-classification-dataset/tzanetakis/tzanetakis/history/"
TZANETAKIS_D_HISTORY_PATH = "/kaggle/input/music-genre-classification-dataset/tzanetakis/tzanetakis/history/"

# 1. Load File Paths

In [ ]:
# Define paths
audio_path = '/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/'
genres = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

# Verify dataset
if os.path.exists(audio_path):
    print("\n✅ GTZAN Dataset found!")
    print(f"Genres: {genres}")
else:
    print("\n❌ Dataset not found! Please add GTZAN dataset from right panel.")


In [ ]:
all_files_path = []
all_genres = []

for i, genre in enumerate(genres):
    genre_path = audio_path + genre
    for _, dirname, files in os.walk(genre_path):
        for file in files:
            all_files_path.append(genre_path + "/" + file)
            all_genres.append(i)

# Split Train Val Test 8:1:1
X_file_train, X_file_test, y_file_train, y_file_test = train_test_split(all_files_path, all_genres, test_size = 0.1, 
                                                                        stratify=all_genres, random_state=42)
X_file_train, X_file_val, y_file_train, y_file_val = train_test_split(X_file_train, y_file_train, test_size = (1/9), 
                                                                      stratify=y_file_train, random_state=42)

# 2. Feature Extraction

## 2.A. Paper 1

In [ ]:
# CELL 27: Audio Segmentation for Data Augmentation
def extract_mfcc_segments(file_path, n_mfcc, segment_duration=3, segment_overlap=0.5, max_pad_len=130):
    """
    Split 30-sec audio into 3-sec segments for 20x more training data
    """
    try:
        audio, sr = librosa.load(file_path, duration=30, mono=True, sr=22050)
        segment_samples = segment_duration * sr
        segments_mfcc = []
        
        for start in range(0, len(audio) - int(segment_samples*(1-segment_overlap)), int(segment_samples*(1-segment_overlap))):
            segment = audio[start:start + segment_samples]
            mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=n_mfcc)
            
            if mfcc.shape[1] < max_pad_len:
                pad_width = max_pad_len - mfcc.shape[1]
                mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
            else:
                mfcc = mfcc[:, :max_pad_len]
            
            segments_mfcc.append(mfcc)
        
        return segments_mfcc
    except Exception as e:
        return []

def extract_mel_segments(file_path, segment_duration=3, n_fft=2048, segment_overlap=0.5, max_pad_len=129):
    """
    Split audio and extract mel-spectrograms from segments
    """
    try:
        audio, sr = librosa.load(file_path, duration=30, mono=True, sr=22050)
        segment_samples = segment_duration * sr
        segments_mel = []
        
        for start in range(0, len(audio) - int(segment_samples*(1-segment_overlap)), int(segment_samples*(1-segment_overlap))):
            segment = audio[start:start + segment_samples]
            mel_spec = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=128,
                                                  n_fft=n_fft, hop_length=512)
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
            
            if mel_spec_db.shape[1] < max_pad_len:
                pad_width = max_pad_len - mel_spec_db.shape[1]
                mel_spec_db = np.pad(mel_spec_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
            else:
                mel_spec_db = mel_spec_db[:, :max_pad_len]
            
            segments_mel.append(mel_spec_db)
        
        return segments_mel
    except Exception as e:
        return []

In [ ]:
def augment_noise(matrix, noise_factor=0.01):
    """Adds random Gaussian noise."""
    noise = np.random.normal(0, matrix.std() * noise_factor, matrix.shape)
    return matrix + noise

def augment_time_shift(matrix, shift_max=20):
    """Randomly shifts in time."""
    shift = np.random.randint(-shift_max, shift_max)
    return np.roll(matrix, shift, axis=1)

def augment_volume(matrix, min_factor=0.9, max_factor=1.1):
    """
    Applies a random volume change (gain) to a 2D matrix.
    """
    volume_factor = np.random.uniform(min_factor, max_factor)
    return matrix * volume_factor

def augment_specaugment(matrix, max_time_mask=20, max_freq_mask=5):
    """Applies Time and Frequency Masking (SpecAugment)."""
    spec = matrix.copy()
    
    # 1. Frequency Masking (Horizontal bar)
    freq_mask_width = np.random.randint(0, max_freq_mask)
    freq_start_point = np.random.randint(0, spec.shape[0] - freq_mask_width)
    spec[freq_start_point:freq_start_point + freq_mask_width, :] = 0

    # 2. Time Masking (Vertical bar)
    time_mask_width = np.random.randint(0, max_time_mask)
    time_start_point = np.random.randint(0, spec.shape[1] - time_mask_width)
    spec[:, time_start_point:time_start_point + time_mask_width] = 0
    
    return spec

In [ ]:
def extract_mfcc_features_from_file_path(X_file, y_file, augmentation, split_type, n_mfcc, segment_overlap, new_class = False, num_classes=10):
    X_aug_mfcc = []
    y_aug_mfcc = []
    genre_mfcc_segment_counts = {i: 0 for i in range(num_classes)}
    if new_class:
        filename = f"{split_type}_mfcc{n_mfcc}_overlap{segment_overlap}_d.npz"
    else:
        filename = f"{split_type}_mfcc{n_mfcc}_overlap{segment_overlap}.npz"
    load_path = MFCC_FEATURES_INPUT_PATH + filename
    save_path = OUTPUT_FEATURES_PATH + filename
    
    if os.path.exists(load_path):
        data = np.load(load_path, allow_pickle=True)
        aug_mfcc = {
            'features': data['features'],
            'labels': data['labels'],
            'keys': data['keys']
        }
        print(f"MFCC Features Loaded. Shape = {aug_mfcc['features'].shape}")
        return aug_mfcc
    
    mfcc_keys = []
    for i, file_path in enumerate(X_file):
        current_genre = y_file[i]
        base = os.path.splitext(os.path.basename(file_path))[0]
        mfcc_segments = extract_mfcc_segments(file_path, n_mfcc=n_mfcc, segment_overlap=segment_overlap)
        
        for idx, segment in enumerate(mfcc_segments):
            key = f"{base}_{idx}"
            X_aug_mfcc.append(segment)
            y_aug_mfcc.append(current_genre)
            mfcc_keys.append(key)
            genre_mfcc_segment_counts[current_genre] += 1
            
            if augmentation:
                aug = augment_specaugment(segment)
                aug_key = f"{base}_{idx}_aug"
                X_aug_mfcc.append(aug)
                y_aug_mfcc.append(current_genre)
                mfcc_keys.append(aug_key)
                genre_mfcc_segment_counts[current_genre] += 1
    
    X_aug_mfcc = np.array(X_aug_mfcc)
    y_aug_mfcc = np.array(y_aug_mfcc)
    y_aug_mfcc = to_categorical(y_aug_mfcc, num_classes=num_classes)
    
    os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)
    np.savez_compressed(save_path, 
                        features=X_aug_mfcc, 
                        labels=y_aug_mfcc, 
                        keys=np.array(mfcc_keys))
    print(f"MFCC Features Saved. Shape = {X_aug_mfcc.shape}")
    
    return {"features": X_aug_mfcc, "labels": y_aug_mfcc, "keys": np.array(mfcc_keys)}


def extract_mel_features_from_file_path(X_file, y_file, augmentation, split_type, n_fft, segment_overlap, new_class = False, num_classes=10):
    X_aug_mel = []
    y_aug_mel = []
    
    genre_mel_segment_counts = {i: 0 for i in range(num_classes)}

    if new_class:
        filename = f"{split_type}_mel{n_fft}_overlap{segment_overlap}_d.npz"
    else:
        filename = f"{split_type}_mel{n_fft}_overlap{segment_overlap}.npz"
    if n_fft == 1024:
        load_path = MEL_1024_FEATURES_INPUT_PATH + filename
    elif n_fft == 2048:
        load_path = MEL_2048_FEATURES_INPUT_PATH + filename
    save_path = OUTPUT_FEATURES_PATH + filename
    
    if os.path.exists(load_path):
        data = np.load(load_path, allow_pickle=True)
        aug_mel = {
            'features': data['features'],
            'labels': data['labels'],
            'keys': data['keys']
        }
        print(f"Mel Features Loaded. Shape = {aug_mel['features'].shape}")
        return aug_mel
    
    mel_keys = []
    for i, file_path in enumerate(X_file):
        current_genre = y_file[i]
        base = os.path.splitext(os.path.basename(file_path))[0]
        mel_segments = extract_mel_segments(file_path, segment_overlap=segment_overlap, n_fft=n_fft)
        
        for idx, segment in enumerate(mel_segments):
            key = f"{base}_{idx}"
            X_aug_mel.append(segment)
            y_aug_mel.append(current_genre)
            mel_keys.append(key)
            genre_mel_segment_counts[current_genre] += 1
            
            if augmentation:
                aug = augment_specaugment(segment)
                aug_key = f"{base}_{idx}_aug"
                X_aug_mel.append(aug)
                y_aug_mel.append(current_genre)
                mel_keys.append(aug_key)
                genre_mel_segment_counts[current_genre] += 1
    
    # Convert to numpy arrays
    X_aug_mel = np.array(X_aug_mel)
    y_aug_mel = np.array(y_aug_mel)
    y_aug_mel = to_categorical(y_aug_mel, num_classes=num_classes)
    
    os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)
    np.savez_compressed(save_path, 
                        features=X_aug_mel, 
                        labels=y_aug_mel, 
                        keys=np.array(mel_keys))
    print(f"Mel Features Saved. Shape = {X_aug_mel.shape}")
    
    return {"features": X_aug_mel, "labels": y_aug_mel, "keys": np.array(mel_keys)}


In [ ]:
class TzanetakisWholeFileExtractor:
    def __init__(self, file_path, sr=22050):
        """
        Initializes extractor with the standard sample rate from the paper (22050 Hz).
        """
        self.sr = sr
        self.y, _ = librosa.load(file_path, sr=self.sr)
        
        # --- Constants from Paper ---
        # Analysis Window: 23 ms (512 samples) [cite: 128]
        self.N_FFT = 512           
        self.HOP_LENGTH = 512      
        
        # Texture Window: 1 second (43 analysis windows) [cite: 128]
        self.TEXTURE_WIN_FRAMES = 43 

    def extract_timbral(self):
        """
        Extracts 19 Timbral features.
        Logic: Compute 1s Texture Windows (Mean/Var) -> Average over whole file.
        """
        # 1. Extract Analysis Features (Frame-by-Frame)
        # Spectral Centroid [cite: 95]
        cent = librosa.feature.spectral_centroid(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        # Spectral Rolloff [cite: 100]
        roll = librosa.feature.spectral_rolloff(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH, roll_percent=0.85)[0]
        # Zero Crossings [cite: 109]
        zcr = librosa.feature.zero_crossing_rate(self.y, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        
        # Spectral Flux [cite: 105]
        S = np.abs(librosa.stft(self.y, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH))
        norm_S = S / (np.sum(S, axis=0, keepdims=True) + 1e-9)
        flux = np.sum((norm_S[:, 1:] - norm_S[:, :-1])**2, axis=0)
        flux = np.pad(flux, (1, 0), mode='constant') # Pad to match length

        # MFCCs (First 5, excluding DC) [cite: 117]
        mfcc = librosa.feature.mfcc(y=self.y, sr=self.sr, n_mfcc=6, hop_length=self.HOP_LENGTH)
        mfccs_no_dc = mfcc[1:6] # Drop index 0 (DC)
        
        # RMS (for Low Energy)
        rms = librosa.feature.rms(y=self.y, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]

        # 2. Apply Texture Window (1 Second Rolling Window)
        # We assume 'Analysis and Texture Window' logic applies here 
        
        # Helper to frame signals into 43-frame windows
        def get_texture_stats(series):
            # Create sliding windows
            frames = librosa.util.frame(series, frame_length=self.TEXTURE_WIN_FRAMES, hop_length=1)
            # Calculate Mean and Variance for each window
            means = np.mean(frames, axis=0)
            vars = np.var(frames, axis=0)
            return means, vars

        # Lists to hold the sequences of Texture Vectors
        texture_vectors = [] 

        # Calculate Texture Stats for standard features
        t_cent_m, t_cent_v = get_texture_stats(cent)
        t_roll_m, t_roll_v = get_texture_stats(roll)
        t_flux_m, t_flux_v = get_texture_stats(flux)
        t_zcr_m,  t_zcr_v  = get_texture_stats(zcr)
        
        # Calculate Texture Stats for MFCCs
        mfcc_stats = []
        for i in range(5):
            m, v = get_texture_stats(mfccs_no_dc[i])
            mfcc_stats.append((m, v))
            
        # Calculate Low Energy per Texture Window [cite: 129-130]
        rms_frames = librosa.util.frame(rms, frame_length=self.TEXTURE_WIN_FRAMES, hop_length=1)
        rms_means = np.mean(rms_frames, axis=0) # Average RMS in each window
        low_energy_seq = []
        for i in range(rms_frames.shape[1]):
            # % of frames in this window < window mean
            le = np.sum(rms_frames[:, i] < rms_means[i]) / self.TEXTURE_WIN_FRAMES
            low_energy_seq.append(le)
        low_energy_seq = np.array(low_energy_seq)

        # 3. Whole File Integration
        # "Mean feature vector over the whole file is used" 
        final_vec = []
        
        # Append means of the texture means/vars
        final_vec.extend([np.mean(t_cent_m), np.mean(t_cent_v)])
        final_vec.extend([np.mean(t_roll_m), np.mean(t_roll_v)])
        final_vec.extend([np.mean(t_flux_m), np.mean(t_flux_v)])
        final_vec.extend([np.mean(t_zcr_m),  np.mean(t_zcr_v)])
        
        for m, v in mfcc_stats:
            final_vec.extend([np.mean(m), np.mean(v)])
            
        final_vec.append(np.mean(low_energy_seq))
        
        return np.array(final_vec)

    def extract_rhythmic(self):
        """
        Extracts 6 Rhythmic features using DWT over the whole file.
        "Peaks are accumulated over the whole sound file into a beat histogram"[cite: 183].
        Window: ~3s (65536 samples), Hop: ~1.5s (32768 samples) [cite: 251-252].
        """
        WINDOW_SIZE = 65536
        HOP_SIZE = 32768
        
        # Global Beat Histogram Accumulator
        # Range 40-200 BPM. We'll map indices later.
        global_histogram_peaks = []
        global_histogram_bpms = []
        
        total_samples = len(self.y)
        
        # Iterate over file in 3s chunks with overlap
        for start in range(0, total_samples, HOP_SIZE):
            end = min(start + WINDOW_SIZE, total_samples)
            y_window = self.y[start:end]
            
            # Need roughly 3s for DWT to work effectively
            if len(y_window) < WINDOW_SIZE * 0.5: 
                continue

            # 1. DWT & Envelope [cite: 178-180]
            coeffs = pywt.wavedec(y_window, 'db4', level=4)
            combined_sum = None
            
            for band in coeffs:
                if len(band) != len(y_window):
                    band = scipy.signal.resample(band, len(y_window))
                # Rectification + LowPass + Downsample [cite: 188-197]
                env = scipy.signal.lfilter([1 - 0.99], [1, -0.99], np.abs(band))
                env_ds = env[::16] - np.mean(env[::16])
                
                if combined_sum is None: combined_sum = env_ds
                else: combined_sum += env_ds[:len(combined_sum)]

            # 2. Autocorrelation [cite: 202]
            corr = np.correlate(combined_sum, combined_sum, mode='full')
            corr = corr[len(corr)//2:]
            
            # 3. Peak Accumulation Logic
            eff_sr = self.sr / 16
            bpms = 60 * eff_sr / (np.arange(len(corr)) + 1e-9)
            
            valid_idx = (bpms >= 40) & (bpms <= 200)
            valid_corr = corr[valid_idx]
            valid_bpms = bpms[valid_idx]
            
            peaks, _ = scipy.signal.find_peaks(valid_corr, height=0)
            
            # Accumulate significant peaks into global list
            for p in peaks:
                global_histogram_peaks.append(valid_corr[p])
                global_histogram_bpms.append(valid_bpms[p])

        # 4. Construct Features from Global Accumulation
        # (Simplified: finding dominant peaks from the accumulated set)
        if len(global_histogram_peaks) == 0:
            return np.zeros(6)

        # Sort all accumulated peaks by amplitude
        sorted_peaks = sorted(zip(global_histogram_peaks, global_histogram_bpms), reverse=True, key=lambda x: x[0])
        
        sum_amp = np.sum(global_histogram_peaks) + 1e-9 # [cite: 250]
        
        a0, p1 = sorted_peaks[0] if len(sorted_peaks) > 0 else (0, 0) # [cite: 247, 249]
        a1, p2 = sorted_peaks[1] if len(sorted_peaks) > 1 else (0, 0)
        
        ra = (a1 / a0) if a0 > 0 else 0 # [cite: 248]
        
        return np.array([a0/sum_amp, a1/sum_amp, ra, p1, p2, sum_amp])

    def extract_pitch(self):
        """
        Extracts 5 Pitch features accumulated over the whole file[cite: 364].
        """
        # 1. Folded Histogram (Chroma)
        # "Accumulated into a PH over the whole sound file" 
        chroma = librosa.feature.chroma_stft(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        fph_raw = np.sum(chroma, axis=1)
        
        # Circle of Fifths Mapping [cite: 283-286]
        fph = np.zeros(12)
        for c in range(12): 
            fph[(c * 7) % 12] = fph_raw[c]
            
        # 2. Unfolded Histogram (Pitch Track)
        pitches, mags = librosa.piptrack(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        uph_vals = [pitches[mags[:, t].argmax(), t] for t in range(pitches.shape[1]) if mags[:, t].max() > 0]
        
        # 3. Feature Calculation [cite: 294-362]
        FA0 = np.max(fph)
        FP0 = np.argmax(fph)
        
        UP0 = 0
        if uph_vals:
            hist, bins = np.histogram(uph_vals, bins=20)
            UP0 = bins[np.argmax(hist)]
            
        sorted_fph = np.argsort(fph)[::-1]
        IPO1 = abs(sorted_fph[0] - sorted_fph[1]) if len(sorted_fph) > 1 else 0
        SUM = np.sum(fph)
        
        return np.array([FA0, UP0, FP0, IPO1, SUM])

    def extract_all(self):
        """
        Returns the single 30-dimensional feature vector for the whole 30s file.
        """
        
        t = self.extract_timbral()   # 19
        r = self.extract_rhythmic()  # 6
        p = self.extract_pitch()     # 5
        
        features = np.concatenate([t, r, p])
        
        
        return features


def build_feature_dataset(file_paths, labels, split_type,sr=22050, new_class=True, num_classes=10):
    """
    Iterates through file paths, extracts Tzanetakis features, and returns X and y arrays.
    """
    X_features = []
    y_labels = []

    if new_class:
        filename = f"{split_type}_tzanetakis_d.npz"
    else:
        filename = f"{split_type}_tzanetakis.npz"
    load_path = TZANETAKIS_FEATURES_INPUT_PATH + filename
    save_path = OUTPUT_FEATURES_PATH + filename
    
    if os.path.exists(load_path):
        data = np.load(load_path, allow_pickle=True)
        tzanetakis = {
            'features': data['features'],
            'labels': data['labels']
        }
        print(f"Tzanetakis Features Loaded. Shape = {tzanetakis['features'].shape}")
        return tzanetakis
    
    # Iterate with index to keep track
    print(f"Processing {len(file_paths)} files...")
    
    for file_path, label in tqdm(zip(file_paths, labels), total=len(file_paths)):
        try:
            # 1. Check if file exists to avoid crashing
            if not os.path.exists(file_path):
                print(f"Warning: File not found {file_path}")
                continue
                
            # 2. Initialize Extractor
            extractor = TzanetakisWholeFileExtractor(file_path, sr=sr)
            
            # 3. Extract Feature Vector (Shape: 30,)
            vector = extractor.extract_all()
            
            # 4. Validation: Ensure vector is valid and has correct shape
            # (Sometimes short audio files return zeros or wrong shapes)
            if vector.shape == (30,):
                X_features.append(vector)
                y_labels.append(label)
            else:
                print(f"Skipping {file_path}: Invalid vector shape {vector.shape}")
                
        except Exception as e:
            # Handle librosa loading errors or math errors safely
            print(f"Error processing {file_path}: {e}")
            continue
    
    X_features = np.array(X_features)
    y_labels = np.array(y_labels)
    y_labels = to_categorical(y_labels, num_classes=num_classes)
    os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)
    np.savez_compressed(save_path, 
                        features=X_features, 
                        labels=y_labels)
            
    return {"features": X_features, "labels": y_labels}

In [ ]:
class TzanetakisMatrixExtractor:
    def __init__(self, file_path, sr=22050):
        self.sr = sr
        self.y_full, _ = librosa.load(file_path, sr=self.sr)
        
        # Paper Constants
        self.N_FFT = 512            # Analysis Window (23ms) [cite: 128]
        self.HOP_LENGTH = 512       # Non-overlapping analysis for texture calculation
        
        # 1 Second Texture Window = 43 Analysis Windows [cite: 128]
        self.TEXTURE_WIN_FRAMES = 43

    # ==========================================
    # 1. Feature Extraction Kernels
    # ==========================================
    def _get_timbral_vector(self, y_1sec):
        """
        Calculates 19-dim timbral vector for a specific 1-second sub-segment.
        """
        # If segment is too short (end of file), return zeros
        if len(y_1sec) < 512:
            return np.zeros(19)

        # Extract Analysis Features (Frame-by-frame)
        cent = librosa.feature.spectral_centroid(y=y_1sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        roll = librosa.feature.spectral_rolloff(y=y_1sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH, roll_percent=0.85)[0]
        zcr  = librosa.feature.zero_crossing_rate(y_1sec, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        
        # Flux (pad to match length)
        S = np.abs(librosa.stft(y_1sec, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH))
        norm_S = S / (np.sum(S, axis=0, keepdims=True) + 1e-9)
        flux = np.sum((norm_S[:, 1:] - norm_S[:, :-1])**2, axis=0)
        flux = np.pad(flux, (1, 0), mode='constant')

        # MFCC (Exclude DC [cite: 133])
        mfcc = librosa.feature.mfcc(y=y_1sec, sr=self.sr, n_mfcc=6, hop_length=self.HOP_LENGTH)
        mfccs_no_dc = mfcc[1:6] 

        # RMS for Low Energy
        rms = librosa.feature.rms(y=y_1sec, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        
        # --- Calculate Gaussian Parameters (Means & Variances) [cite: 124-125] ---
        # Since input y_1sec is exactly 1 texture window size, we just take the global stats of this slice.
        feats = []
        for series in [cent, roll, flux, zcr]:
            feats.extend([np.mean(series), np.var(series)])
            
        for i in range(5):
            feats.extend([np.mean(mfccs_no_dc[i]), np.var(mfccs_no_dc[i])])
            
        # Low Energy [cite: 129-130]
        avg_rms = np.mean(rms)
        low_e = np.sum(rms < avg_rms) / (len(rms) + 1e-9)
        feats.append(low_e)
        
        return np.array(feats) # Shape (19,)

    def _get_rhythmic_vector(self, y_3sec):
        """
        Calculates 6-dim rhythmic vector for the full 3-second context.
        [cite: 247-250]
        """
        coeffs = pywt.wavedec(y_3sec, 'db4', level=3)
        combined_sum = None
        for band in coeffs:
            if len(band) != len(y_3sec):
                 band = scipy.signal.resample(band, len(y_3sec))
            # Rectification + Low Pass + Downsampling [cite: 188-197]
            env = scipy.signal.lfilter([1 - 0.99], [1, -0.99], np.abs(band))
            env_ds = env[::16] - np.mean(env[::16])
            if combined_sum is None: combined_sum = env_ds
            else: combined_sum += env_ds[:len(combined_sum)]

        corr = np.correlate(combined_sum, combined_sum, mode='full')
        corr = corr[len(corr)//2:]
        bpms = 60 * (self.sr / 16) / (np.arange(len(corr)) + 1e-9)
        
        valid_idx = (bpms >= 40) & (bpms <= 200)
        valid_corr = corr[valid_idx]
        valid_bpms = bpms[valid_idx]
        
        peaks, _ = scipy.signal.find_peaks(valid_corr, height=0)
        if len(peaks) == 0: return np.zeros(6)
            
        # Sort peaks by amplitude
        sorted_peaks = sorted(zip(valid_corr[peaks], valid_bpms[peaks]), reverse=True, key=lambda x: x[0])
        
        sum_amp = np.sum(valid_corr) + 1e-9
        a0, p1 = sorted_peaks[0] if len(sorted_peaks) > 0 else (0, 0)
        a1, p2 = sorted_peaks[1] if len(sorted_peaks) > 1 else (0, 0)
        
        return np.array([a0/sum_amp, a1/sum_amp, (a1/a0) if a0>0 else 0, p1, p2, sum_amp])

    def _get_pitch_vector(self, y_3sec):
        """
        Calculates 5-dim pitch vector for the full 3-second context.
        [cite: 294-362]
        """
        # Folded (Chroma)
        chroma = librosa.feature.chroma_stft(y=y_3sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        fph_raw = np.sum(chroma, axis=1)
        fph = np.zeros(12)
        for c in range(12): fph[(c * 7) % 12] = fph_raw[c] # Circle of fifths mapping
            
        # Unfolded (Pitch Track)
        pitches, mags = librosa.piptrack(y=y_3sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        uph_vals = [pitches[mags[:, t].argmax(), t] for t in range(pitches.shape[1]) if mags[:, t].max() > 0]
        
        UP0 = 0
        if uph_vals:
            hist, bins = np.histogram(uph_vals, bins=20)
            UP0 = bins[np.argmax(hist)]
            
        sorted_fph = np.argsort(fph)[::-1]
        IPO1 = abs(sorted_fph[0] - sorted_fph[1]) if len(sorted_fph) > 1 else 0
        
        return np.array([np.max(fph), UP0, np.argmax(fph), IPO1, np.sum(fph)])

    # ==========================================
    # 2. Main Processing Logic
    # ==========================================
    def process_file(self):
        """
        Returns a (30, 30) matrix for a 30-second file.
        Structure: 10 segments * 3 vectors_per_segment.
        """
        # Define 3-second segments
        samples_3s = int(3.0 * self.sr)
        samples_1s = int(1.0 * self.sr)
        
        all_vectors = []

        # Iterate 10 times (or until file ends)
        total_samples = len(self.y_full)
        
        for start_3s in range(0, total_samples, samples_3s):
            end_3s = min(start_3s + samples_3s, total_samples)
            y_segment_3s = self.y_full[start_3s:end_3s]
            
            # Skip if segment is too short to be meaningful
            if len(y_segment_3s) < self.sr:
                continue

            # --- A. Macro Features (Broadcasting) ---
            # Calculated ONCE per 3s segment
            r_vec = self._get_rhythmic_vector(y_segment_3s) # 6 dims
            p_vec = self._get_pitch_vector(y_segment_3s)    # 5 dims
            macro_vec = np.concatenate([r_vec, p_vec])      # 11 dims

            # --- B. Micro Features (Splitting) ---
            # Split the 3s segment into three 1s sub-segments
            # This generates the 3 rows for this segment
            for i in range(3):
                start_1s = i * samples_1s
                end_1s = min(start_1s + samples_1s, len(y_segment_3s))
                
                # If we ran out of data in the last split (e.g. file ends exactly at 3s), use zeros
                if start_1s >= len(y_segment_3s):
                    t_vec = np.zeros(19)
                else:
                    y_sub_1s = y_segment_3s[start_1s:end_1s]
                    t_vec = self._get_timbral_vector(y_sub_1s) # 19 dims
                
                # --- C. Concatenate ---
                # [Timbral (19) | Rhythmic (6) | Pitch (5)] = 30 dims
                full_vec = np.concatenate([t_vec, macro_vec])
                all_vectors.append(full_vec)

        # Convert to numpy array
        result_matrix = np.array(all_vectors)
        
        # Ensure we return exactly (30, 30) if the file was long enough
        # (Truncate or pad if necessary for consistency)
        if result_matrix.shape[0] > 30:
            result_matrix = result_matrix[:30]
        elif result_matrix.shape[0] < 30:
            # Pad with zeros if file was short
            pad_rows = 30 - result_matrix.shape[0]
            result_matrix = np.pad(result_matrix, ((0, pad_rows), (0, 0)), mode='constant')
            
        return result_matrix

def build_matrix_dataset(file_paths, labels, split_type, sr=22050, new_class=True, num_classes=10):
    """
    Iterates through file paths using TzanetakisMatrixExtractor.
    Returns X as a 3D tensor: (N_files, 30_steps, 30_features).
    """
    X_matrices = []
    y_labels = []
    
    if new_class:
        filename = f"{split_type}_tzanetakis_m_d.npz"
    else:
        filename = f"{split_type}_tzanetakis_m.npz"
    load_path = TZANETAKIS_FEATURES_INPUT_PATH + filename
    save_path = OUTPUT_FEATURES_PATH + filename
    
    if os.path.exists(load_path):
        data = np.load(load_path, allow_pickle=True)
        tzanetakis = {
            'features': data['features'],
            'labels': data['labels']
        }
        print(f"Tzanetakis Features Loaded. Shape = {tzanetakis['features'].shape}")
        return tzanetakis
    
    print(f"Processing {len(file_paths)} files...")
    
    # Using enumerate/zip to keep track of files
    for file_path, label in tqdm(zip(file_paths, labels), total=len(file_paths)):
        try:
            if not os.path.exists(file_path):
                print(f"Warning: File not found {file_path}")
                continue
                
            # Initialize Matrix Extractor
            extractor = TzanetakisMatrixExtractor(file_path, sr=sr)
            
            # Extract Matrix (Shape: 30, 30)
            matrix = extractor.process_file()
            
            # Validation: Ensure output is exactly (30, 30)
            if matrix.shape == (30, 30):
                X_matrices.append(matrix)
                y_labels.append(label)
            else:
                print(f"Skipping {file_path}: Invalid shape {matrix.shape}")
                
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue
        
    X_matrices = np.array(X_matrices)
    y_labels = np.array(y_labels)
    y_labels = to_categorical(y_labels, num_classes=num_classes)
    
    os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)
    np.savez_compressed(save_path, 
                        features=X_matrices, 
                        labels=y_labels)
            
    return {"features": X_matrices, "labels": y_labels}

In [ ]:
# Augmenting Dataset

class TzanetakisWholeFileAugExtractor:
    def __init__(self, input_data, sr=22050, is_path=False):
        """
        Modified to accept either a file path OR a raw audio array.
        """
        self.sr = sr
        
        # Logic to handle path vs raw data
        if is_path:
            self.y, _ = librosa.load(input_data, sr=self.sr)
        else:
            self.y = input_data # Assume input_data is already a numpy array (3s segment)
        
        # --- Constants from Paper ---
        self.N_FFT = 512            
        self.HOP_LENGTH = 512       
        self.TEXTURE_WIN_FRAMES = 43 

    def extract_timbral(self):
        # 1. Extract Analysis Features
        cent = librosa.feature.spectral_centroid(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        roll = librosa.feature.spectral_rolloff(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH, roll_percent=0.85)[0]
        zcr = librosa.feature.zero_crossing_rate(self.y, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        
        S = np.abs(librosa.stft(self.y, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH))
        norm_S = S / (np.sum(S, axis=0, keepdims=True) + 1e-9)
        flux = np.sum((norm_S[:, 1:] - norm_S[:, :-1])**2, axis=0)
        flux = np.pad(flux, (1, 0), mode='constant')

        mfcc = librosa.feature.mfcc(y=self.y, sr=self.sr, n_mfcc=6, hop_length=self.HOP_LENGTH)
        mfccs_no_dc = mfcc[1:6] 
        
        # Fixed rms call
        rms = librosa.feature.rms(y=self.y, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]

        # 2. Apply Texture Window
        def get_texture_stats(series):
            # Check for short segments
            if len(series) < self.TEXTURE_WIN_FRAMES:
                return np.array([np.mean(series)]), np.array([np.var(series)])
            
            frames = librosa.util.frame(series, frame_length=self.TEXTURE_WIN_FRAMES, hop_length=1)
            means = np.mean(frames, axis=0)
            vars = np.var(frames, axis=0)
            return means, vars

        t_cent_m, t_cent_v = get_texture_stats(cent)
        t_roll_m, t_roll_v = get_texture_stats(roll)
        t_flux_m, t_flux_v = get_texture_stats(flux)
        t_zcr_m,  t_zcr_v  = get_texture_stats(zcr)
        
        mfcc_stats = []
        for i in range(5):
            m, v = get_texture_stats(mfccs_no_dc[i])
            mfcc_stats.append((m, v))
            
        # Low Energy
        if len(rms) < self.TEXTURE_WIN_FRAMES:
             low_energy_seq = [0.0]
        else:
            rms_frames = librosa.util.frame(rms, frame_length=self.TEXTURE_WIN_FRAMES, hop_length=1)
            rms_means = np.mean(rms_frames, axis=0) 
            low_energy_seq = []
            for i in range(rms_frames.shape[1]):
                le = np.sum(rms_frames[:, i] < rms_means[i]) / self.TEXTURE_WIN_FRAMES
                low_energy_seq.append(le)
            low_energy_seq = np.array(low_energy_seq)

        # 3. Whole File Integration
        final_vec = []
        final_vec.extend([np.mean(t_cent_m), np.mean(t_cent_v)])
        final_vec.extend([np.mean(t_roll_m), np.mean(t_roll_v)])
        final_vec.extend([np.mean(t_flux_m), np.mean(t_flux_v)])
        final_vec.extend([np.mean(t_zcr_m),  np.mean(t_zcr_v)])
        
        for m, v in mfcc_stats:
            final_vec.extend([np.mean(m), np.mean(v)])
            
        final_vec.append(np.mean(low_energy_seq))
        return np.array(final_vec)

    def extract_rhythmic(self):
        # Adjusted constants for smaller segments (3s)
        # 3s = ~66k samples. The original code's window size fits exactly one 3s segment.
        WINDOW_SIZE = 65536 
        
        # If the segment is exactly 3s, we just process the whole thing as one window
        # For simplicity, we process whatever 'self.y' is provided
        
        coeffs = pywt.wavedec(self.y, 'db4', level=4)
        combined_sum = None
        
        for band in coeffs:
            if len(band) != len(self.y):
                band = scipy.signal.resample(band, len(self.y))
            env = scipy.signal.lfilter([1 - 0.99], [1, -0.99], np.abs(band))
            env_ds = env[::16] - np.mean(env[::16])
            
            if combined_sum is None: combined_sum = env_ds
            else: combined_sum += env_ds[:len(combined_sum)]

        corr = np.correlate(combined_sum, combined_sum, mode='full')
        corr = corr[len(corr)//2:]
        
        eff_sr = self.sr / 16
        bpms = 60 * eff_sr / (np.arange(len(corr)) + 1e-9)
        
        valid_idx = (bpms >= 40) & (bpms <= 200)
        valid_corr = corr[valid_idx]
        valid_bpms = bpms[valid_idx]
        
        peaks, _ = scipy.signal.find_peaks(valid_corr, height=0)
        
        # Accumulation Logic (Local to this segment)
        local_peaks = []
        local_bpms = []
        for p in peaks:
            local_peaks.append(valid_corr[p])
            local_bpms.append(valid_bpms[p])

        if len(local_peaks) == 0: return np.zeros(6)

        sorted_peaks = sorted(zip(local_peaks, local_bpms), reverse=True, key=lambda x: x[0])
        sum_amp = np.sum(local_peaks) + 1e-9 
        
        a0, p1 = sorted_peaks[0] if len(sorted_peaks) > 0 else (0, 0)
        a1, p2 = sorted_peaks[1] if len(sorted_peaks) > 1 else (0, 0)
        ra = (a1 / a0) if a0 > 0 else 0 
        
        return np.array([a0/sum_amp, a1/sum_amp, ra, p1, p2, sum_amp])

    def extract_pitch(self):
        chroma = librosa.feature.chroma_stft(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        fph_raw = np.sum(chroma, axis=1)
        fph = np.zeros(12)
        for c in range(12): fph[(c * 7) % 12] = fph_raw[c]
            
        pitches, mags = librosa.piptrack(y=self.y, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        uph_vals = [pitches[mags[:, t].argmax(), t] for t in range(pitches.shape[1]) if mags[:, t].max() > 0]
        
        FA0 = np.max(fph)
        FP0 = np.argmax(fph)
        UP0 = 0
        if uph_vals:
            hist, bins = np.histogram(uph_vals, bins=20)
            UP0 = bins[np.argmax(hist)]
            
        sorted_fph = np.argsort(fph)[::-1]
        IPO1 = abs(sorted_fph[0] - sorted_fph[1]) if len(sorted_fph) > 1 else 0
        SUM = np.sum(fph)
        
        return np.array([FA0, UP0, FP0, IPO1, SUM])

    def extract_all(self):
        t = self.extract_timbral()   # 19
        r = self.extract_rhythmic()  # 6
        p = self.extract_pitch()     # 5
        return np.concatenate([t, r, p])

# ==========================================
# Modified Dataset Builder
# ==========================================
def build_t_aug_feature_dataset(file_paths, labels, split_type, sr=22050, overlap = 0.0, new_class=True, num_classes=10):
    """
    1. Loads the whole file once.
    2. Splits it into non-overlapping 3s segments.
    3. Extracts features for each segment.
    """
    X_features = []
    y_labels = []
    if new_class:
        filename = f"{split_type}_tzanetakis_aug_d.npz"
    else:
        filename = f"{split_type}_tzanetakis_aug.npz"
    load_path = TZANETAKIS_FEATURES_INPUT_PATH + filename
    save_path = OUTPUT_FEATURES_PATH + filename
    
    if os.path.exists(load_path):
        data = np.load(load_path, allow_pickle=True)
        tzanetakis = {
            'features': data['features'],
            'labels': data['labels']
        }
        print(f"Tzanetakis Features Loaded. Shape = {tzanetakis['features'].shape}")
        return tzanetakis
    
    samples_per_segment = int(3.0 * sr) # 3 Seconds
    
    print(f"Processing {len(file_paths)} files (Segmentation Mode)...")
    
    for file_path, label in tqdm(zip(file_paths, labels), total=len(file_paths)):
        try:
            if not os.path.exists(file_path):
                print(f"Warning: File not found {file_path}")
                continue
            
            # 1. Load the full file once
            y_full, _ = librosa.load(file_path, sr=sr)
            total_samples = len(y_full)
            
            # 2. Loop through in 3s chunks (No Overlap)
            for start in range(0, total_samples, samples_per_segment - int(samples_per_segment * overlap)):
                end = start + samples_per_segment
                
                # Check for bounds
                if end > total_samples:
                    # Option A: Discard incomplete end segment (Standard)
                    # continue 
                    
                    # Option B: Pad incomplete segment (To maximize data)
                    y_seg = y_full[start:]
                    if len(y_seg) < sr: continue # Skip if < 1s (garbage)
                    y_seg = np.pad(y_seg, (0, samples_per_segment - len(y_seg)))
                else:
                    y_seg = y_full[start:end]
                
                # 3. Initialize Extractor with RAW DATA (is_path=False)
                extractor = TzanetakisWholeFileAugExtractor(y_seg, sr=sr, is_path=False)
                
                # 4. Extract
                vector = extractor.extract_all()
                
                if vector.shape == (30,):
                    X_features.append(vector)
                    # IMPORTANT: Append the SAME label for this new segment
                    y_labels.append(label)

                
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue

    X_features = np.array(X_features)
    y_labels = np.array(y_labels)
    y_labels = to_categorical(y_labels, num_classes=num_classes)
    
    os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)
    np.savez_compressed(save_path, 
                        features=X_features, 
                        labels=y_labels)
            
    return {"features": X_features, "labels": y_labels}

In [ ]:
class TzanetakisMatrixAugExtractor:
    def __init__(self, input_data, sr=22050, is_path=False):
        """
        Modified to accept either a file path OR a raw audio array.
        """
        self.sr = sr
        
        # Logic to handle path vs raw data
        if is_path:
            self.y_full, _ = librosa.load(input_data, sr=self.sr)
        else:
            self.y_full = input_data # Assume input_data is already a numpy array (3s segment)
        
        # Paper Constants
        self.N_FFT = 512            
        self.HOP_LENGTH = 512       
        self.TEXTURE_WIN_FRAMES = 43

    # ==========================================
    # Feature Extraction Kernels (Unchanged logic)
    # ==========================================
    def _get_timbral_vector(self, y_1sec):
        if len(y_1sec) < 512: return np.zeros(19)
        cent = librosa.feature.spectral_centroid(y=y_1sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        roll = librosa.feature.spectral_rolloff(y=y_1sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH, roll_percent=0.85)[0]
        zcr  = librosa.feature.zero_crossing_rate(y_1sec, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        S = np.abs(librosa.stft(y_1sec, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH))
        norm_S = S / (np.sum(S, axis=0, keepdims=True) + 1e-9)
        flux = np.sum((norm_S[:, 1:] - norm_S[:, :-1])**2, axis=0)
        flux = np.pad(flux, (1, 0), mode='constant')
        mfcc = librosa.feature.mfcc(y=y_1sec, sr=self.sr, n_mfcc=6, hop_length=self.HOP_LENGTH)
        mfccs_no_dc = mfcc[1:6] 
        rms = librosa.feature.rms(y=y_1sec, frame_length=self.N_FFT, hop_length=self.HOP_LENGTH)[0]
        
        feats = []
        for series in [cent, roll, flux, zcr]:
            feats.extend([np.mean(series), np.var(series)])
        for i in range(5):
            feats.extend([np.mean(mfccs_no_dc[i]), np.var(mfccs_no_dc[i])])
        avg_rms = np.mean(rms)
        low_e = np.sum(rms < avg_rms) / (len(rms) + 1e-9)
        feats.append(low_e)
        return np.array(feats)

    def _get_rhythmic_vector(self, y_3sec):
        # ... (Same logic as before) ...
        coeffs = pywt.wavedec(y_3sec, 'db4', level=3)
        combined_sum = None
        for band in coeffs:
            if len(band) != len(y_3sec): band = scipy.signal.resample(band, len(y_3sec))
            env = scipy.signal.lfilter([1 - 0.99], [1, -0.99], np.abs(band))
            env_ds = env[::16] - np.mean(env[::16])
            if combined_sum is None: combined_sum = env_ds
            else: combined_sum += env_ds[:len(combined_sum)]

        corr = np.correlate(combined_sum, combined_sum, mode='full')
        corr = corr[len(corr)//2:]
        bpms = 60 * (self.sr / 16) / (np.arange(len(corr)) + 1e-9)
        valid_idx = (bpms >= 40) & (bpms <= 200)
        valid_corr = corr[valid_idx]
        valid_bpms = bpms[valid_idx]
        peaks, _ = scipy.signal.find_peaks(valid_corr, height=0)
        if len(peaks) == 0: return np.zeros(6)
        sorted_peaks = sorted(zip(valid_corr[peaks], valid_bpms[peaks]), reverse=True, key=lambda x: x[0])
        sum_amp = np.sum(valid_corr) + 1e-9
        a0, p1 = sorted_peaks[0] if len(sorted_peaks) > 0 else (0, 0)
        a1, p2 = sorted_peaks[1] if len(sorted_peaks) > 1 else (0, 0)
        return np.array([a0/sum_amp, a1/sum_amp, (a1/a0) if a0>0 else 0, p1, p2, sum_amp])

    def _get_pitch_vector(self, y_3sec):
        # ... (Same logic as before) ...
        chroma = librosa.feature.chroma_stft(y=y_3sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        fph_raw = np.sum(chroma, axis=1)
        fph = np.zeros(12)
        for c in range(12): fph[(c * 7) % 12] = fph_raw[c]
        pitches, mags = librosa.piptrack(y=y_3sec, sr=self.sr, n_fft=self.N_FFT, hop_length=self.HOP_LENGTH)
        uph_vals = [pitches[mags[:, t].argmax(), t] for t in range(pitches.shape[1]) if mags[:, t].max() > 0]
        UP0 = 0
        if uph_vals:
            hist, bins = np.histogram(uph_vals, bins=20)
            UP0 = bins[np.argmax(hist)]
        sorted_fph = np.argsort(fph)[::-1]
        IPO1 = abs(sorted_fph[0] - sorted_fph[1]) if len(sorted_fph) > 1 else 0
        return np.array([np.max(fph), UP0, np.argmax(fph), IPO1, np.sum(fph)])

    # ==========================================
    # Main Processing Logic (Modified for Single 3s Segment)
    # ==========================================
    def process_segment(self):
        """
        Returns a (3, 30) matrix for the provided audio segment (assumed ~3s).
        Structure: 3 sub-segments (1s) * 30 features.
        """
        y_segment_3s = self.y_full # Input is already just the 3s segment
        
        # --- A. Macro Features (Broadcasting) ---
        # Calculated ONCE per 3s segment
        r_vec = self._get_rhythmic_vector(y_segment_3s) # 6 dims
        p_vec = self._get_pitch_vector(y_segment_3s)    # 5 dims
        macro_vec = np.concatenate([r_vec, p_vec])      # 11 dims

        segment_vectors = []
        samples_1s = int(1.0 * self.sr)

        # --- B. Micro Features (Splitting) ---
        # Split the 3s segment into three 1s sub-segments
        for i in range(3):
            start_1s = i * samples_1s
            end_1s = min(start_1s + samples_1s, len(y_segment_3s))
            
            if start_1s >= len(y_segment_3s):
                t_vec = np.zeros(19)
            else:
                y_sub_1s = y_segment_3s[start_1s:end_1s]
                t_vec = self._get_timbral_vector(y_sub_1s) # 19 dims
            
            # --- C. Concatenate ---
            full_vec = np.concatenate([t_vec, macro_vec])
            segment_vectors.append(full_vec)

        return np.array(segment_vectors) # Shape (3, 30)

# ==========================================
# Modified Dataset Builder (Segmentation Mode)
# ==========================================
def build_matrix_aug_dataset(file_paths, labels, split_type, sr=22050, overlap = 0.0, new_class=False, num_classes=10):
    """
    1. Loads full file.
    2. Splits into 3s segments.
    3. Extracts (3, 30) matrix for each segment.
    """
    X_matrices = []
    y_labels = []

    if new_class:
        filename = f"{split_type}_tzanetakis_m_aug_d.npz"
    else:
        filename = f"{split_type}_tzanetakis_m_aug.npz"
    load_path = TZANETAKIS_FEATURES_INPUT_PATH + filename
    save_path = OUTPUT_FEATURES_PATH + filename
    
    if os.path.exists(load_path):
        data = np.load(load_path, allow_pickle=True)
        tzanetakis = {
            'features': data['features'],
            'labels': data['labels']
        }
        print(f"Tzanetakis Features Loaded. Shape = {tzanetakis['features'].shape}")
        return tzanetakis
    
    samples_per_segment = int(3.0 * sr) # 3 Seconds
    
    print(f"Processing {len(file_paths)} files (Matrix Segmentation Mode)...")
    
    for file_path, label in tqdm(zip(file_paths, labels), total=len(file_paths)):
        try:
            if not os.path.exists(file_path):
                print(f"Warning: File not found {file_path}")
                continue
            
            # 1. Load the full file once
            y_full, _ = librosa.load(file_path, sr=sr)
            total_samples = len(y_full)
            
            # 2. Loop through in 3s chunks (No Overlap)
            for start in range(0, total_samples, samples_per_segment - int(samples_per_segment * overlap)):
                end = start + samples_per_segment
                
                # Check for bounds / Padding
                if end > total_samples:
                    y_seg = y_full[start:]
                    if len(y_seg) < sr: continue # Skip if < 1s
                    y_seg = np.pad(y_seg, (0, samples_per_segment - len(y_seg)))
                else:
                    y_seg = y_full[start:end]
                
                # 3. Initialize Extractor with RAW DATA
                extractor = TzanetakisMatrixAugExtractor(y_seg, sr=sr, is_path=False)
                
                # 4. Extract Matrix (Shape: 3, 30)
                matrix = extractor.process_segment()
                
                if matrix.shape == (3, 30):
                    X_matrices.append(matrix)
                    y_labels.append(label)
                else:
                    print(f"Skipping segment in {file_path}: Invalid shape {matrix.shape}")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue
    
    X_matrices = np.array(X_matrices)
    y_labels = np.array(y_labels)
    y_labels = to_categorical(y_labels, num_classes=num_classes)
    os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)
    np.savez_compressed(save_path, 
                        features=X_matrices, 
                        labels=y_labels)
            
    return {"features": X_matrices, "labels": y_labels}

In [ ]:
print("🔃 Extracting MFCC Features for X train...")
train_aug_mfcc = extract_mfcc_features_from_file_path(X_file_train, y_file_train, augmentation=False, split_type="train", n_mfcc=20, segment_overlap=0.5)
X_train_aug_mfcc = train_aug_mfcc['features']
y_train_aug_mfcc = train_aug_mfcc['labels']

print("🔃 Extracting Mel Features (n_fft = 1024) for X train...")
train_aug_mel_1024 = extract_mel_features_from_file_path(X_file_train, y_file_train, augmentation=False, split_type="train", n_fft=1024, segment_overlap=0.5)
X_train_aug_mel_1024 = train_aug_mel_1024['features']
y_train_aug_mel_1024 = train_aug_mel_1024['labels']

print("🔃 Extracting Mel Features (n_fft = 2048) for X train...")
train_aug_mel_2048 = extract_mel_features_from_file_path(X_file_train, y_file_train, augmentation=False, split_type="train", n_fft=2048, segment_overlap=0.5)
X_train_aug_mel_2048 = train_aug_mel_2048['features']
y_train_aug_mel_2048 = train_aug_mel_2048['labels']

print("🔃 Extracting Tzanetakis Features for X train...")
train_t = build_feature_dataset(X_file_train, y_file_train, split_type='train')
X_train_t = train_t['features']
y_train_t = train_t['labels']

print("🔃 Extracting Tzanetakis Matrix Features for X train...")
train_m = build_matrix_dataset(X_file_train, y_file_train, split_type='train')
X_train_m = train_m['features']
y_train_m = train_m['labels']

print("🔃 Extracting Tzanetakis Augmented Features for X train...")
train_t_aug = build_t_aug_feature_dataset(X_file_train, y_file_train, split_type="train")
X_train_t_aug = train_t_aug['features']
y_train_t_aug = train_t_aug['labels']

print("🔃 Extracting Tzanetakis Matrix Augmented Features for X train...")
train_m_aug = build_matrix_aug_dataset(X_file_train, y_file_train, split_type="train")
X_train_m_aug = train_m_aug['features']
y_train_m_aug = train_m_aug['labels']

In [ ]:
print("🔃 Extracting MFCC Features for X val...")
val_aug_mfcc = extract_mfcc_features_from_file_path(X_file_val, y_file_val, augmentation=False, split_type="val", n_mfcc=20, segment_overlap=0.5)
X_val_aug_mfcc = val_aug_mfcc['features']
y_val_aug_mfcc = val_aug_mfcc['labels']

print("🔃 Extracting Mel Features (n_fft = 1024) for X val...")
val_aug_mel_1024 = extract_mel_features_from_file_path(X_file_val, y_file_val, augmentation=False, split_type="val", n_fft=1024, segment_overlap=0.5)
X_val_aug_mel_1024 = val_aug_mel_1024['features']
y_val_aug_mel_1024 = val_aug_mel_1024['labels']

print("🔃 Extracting Mel Features (n_fft = 2048) for X val...")
val_aug_mel_2048 = extract_mel_features_from_file_path(X_file_val, y_file_val, augmentation=False, split_type="val", n_fft=2048, segment_overlap=0.5)
X_val_aug_mel_2048 = val_aug_mel_2048['features']
y_val_aug_mel_2048 = val_aug_mel_2048['labels']

print("🔃 Extracting Tzanetakis Features for X val...")
val_t = build_feature_dataset(X_file_val, y_file_val, split_type='val')
X_val_t = val_t['features']
y_val_t = val_t['labels']

print("🔃 Extracting Tzanetakis Matrix Features for X val...")
val_m = build_matrix_dataset(X_file_val, y_file_val, split_type='val')
X_val_m = val_m['features']
y_val_m = val_m['labels']

print("🔃 Extracting Tzanetakis Augmented Features for X val...")
val_t_aug = build_t_aug_feature_dataset(X_file_val, y_file_val, split_type="val")
X_val_t_aug = val_t_aug['features']
y_val_t_aug = val_t_aug['labels']

print("🔃 Extracting Tzanetakis Matrix Augmented Features for X val...")
val_m_aug = build_matrix_aug_dataset(X_file_val, y_file_val, split_type="val")
X_val_m_aug = val_m_aug['features']
y_val_m_aug = val_m_aug['labels']

In [ ]:
print("🔃 Extracting MFCC Features for X test...")
test_aug_mfcc = extract_mfcc_features_from_file_path(X_file_test, y_file_test, augmentation=False, split_type="test", n_mfcc=20, segment_overlap=0.5)
X_test_aug_mfcc = test_aug_mfcc['features']
y_test_aug_mfcc = test_aug_mfcc['labels']

print("🔃 Extracting Mel Features (n_fft = 1024) for X test...")
test_aug_mel_1024 = extract_mel_features_from_file_path(X_file_test, y_file_test, augmentation=False, split_type="test", n_fft=1024, segment_overlap=0.5)
X_test_aug_mel_1024 = test_aug_mel_1024['features']
y_test_aug_mel_1024 = test_aug_mel_1024['labels']

print("🔃 Extracting Mel Features (n_fft = 2048) for X test...")
test_aug_mel_2048 = extract_mel_features_from_file_path(X_file_test, y_file_test, augmentation=False, split_type="test", n_fft=2048, segment_overlap=0.5)
X_test_aug_mel_2048 = test_aug_mel_2048['features']
y_test_aug_mel_2048 = test_aug_mel_2048['labels']

print("🔃 Extracting Tzanetakis Features for X test...")
test_t = build_feature_dataset(X_file_test, y_file_test, split_type='test')
X_test_t = test_t['features']
y_test_t = test_t['labels']

print("🔃 Extracting Tzanetakis Matrix Features for X test...")
test_m = build_matrix_dataset(X_file_test, y_file_test, split_type='test')
X_test_m = test_m['features']
y_test_m = test_m['labels']

print("🔃 Extracting Tzanetakis Augmented Features for X test...")
test_t_aug = build_t_aug_feature_dataset(X_file_test, y_file_test, split_type="test")
X_test_t_aug = test_t_aug['features']
y_test_t_aug = test_t_aug['labels']

print("🔃 Extracting Tzanetakis Matrix Augmented Features for X test...")
test_m_aug = build_matrix_aug_dataset(X_file_test, y_file_test, split_type="test")
X_test_m_aug = test_m_aug['features']
y_test_m_aug = test_m_aug['labels']

In [ ]:
X_train_aug_mel_2048_combined = np.vstack([X_train_aug_mel_2048, X_val_aug_mel_2048])
y_train_aug_mel_2048_combined = np.vstack([y_train_aug_mel_2048, y_val_aug_mel_2048])

X_train_aug_mel_1024_combined = np.vstack([X_train_aug_mel_1024, X_val_aug_mel_1024])
y_train_aug_mel_1024_combined = np.vstack([y_train_aug_mel_1024, y_val_aug_mel_1024])

X_test_aug_mel_1024_combined = X_test_aug_mel_1024
y_test_aug_mel_1024_combined = y_test_aug_mel_1024

X_test_aug_mel_2048_combined = X_test_aug_mel_2048
y_test_aug_mel_2048_combined = y_test_aug_mel_2048

X_train_t_aug_combined = np.vstack([X_train_t_aug, X_val_t_aug])
y_train_t_aug_combined = np.vstack([y_train_t_aug, y_val_t_aug])

X_test_t_aug_combined = X_test_t_aug
y_test_t_aug_combined = y_test_t_aug

In [ ]:
# Normalize
X_train_aug_mfcc_reshaped = X_train_aug_mfcc.reshape(X_train_aug_mfcc.shape[0], -1)
X_test_aug_mfcc_reshaped = X_test_aug_mfcc.reshape(X_test_aug_mfcc.shape[0], -1)
X_val_aug_mfcc_reshaped = X_val_aug_mfcc.reshape(X_val_aug_mfcc.shape[0], -1)
scaler_aug_mfcc = StandardScaler()
X_train_aug_mfcc_scaled = scaler_aug_mfcc.fit_transform(X_train_aug_mfcc_reshaped)
X_test_aug_mfcc_scaled = scaler_aug_mfcc.transform(X_test_aug_mfcc_reshaped)
X_val_aug_mfcc_scaled = scaler_aug_mfcc.transform(X_val_aug_mfcc_reshaped)
X_train_aug_mfcc = X_train_aug_mfcc_scaled.reshape(X_train_aug_mfcc.shape[0], X_train_aug_mfcc.shape[1], X_train_aug_mfcc.shape[2])
X_val_aug_mfcc = X_val_aug_mfcc_scaled.reshape(X_val_aug_mfcc.shape[0], X_val_aug_mfcc.shape[1], X_val_aug_mfcc.shape[2])
X_test_aug_mfcc = X_test_aug_mfcc_scaled.reshape(X_test_aug_mfcc.shape[0], X_test_aug_mfcc.shape[1], X_test_aug_mfcc.shape[2])

# Reshape for 2D-CNN
X_train_aug_mfcc = X_train_aug_mfcc.reshape(X_train_aug_mfcc.shape[0], X_train_aug_mfcc.shape[1], X_train_aug_mfcc.shape[2], 1)
X_val_aug_mfcc = X_val_aug_mfcc.reshape(X_val_aug_mfcc.shape[0], X_val_aug_mfcc.shape[1], X_val_aug_mfcc.shape[2], 1)
X_test_aug_mfcc = X_test_aug_mfcc.reshape(X_test_aug_mfcc.shape[0], X_test_aug_mfcc.shape[1], X_test_aug_mfcc.shape[2], 1)

print(f"Training shape: {X_train_aug_mfcc.shape}")
print(f"Validation shape: {X_val_aug_mfcc.shape}")
print(f"Test shape: {X_test_aug_mfcc.shape}")

In [ ]:
# Normalize 1024
X_train_aug_mel_1024_reshaped = X_train_aug_mel_1024.reshape(X_train_aug_mel_1024.shape[0], -1)
X_test_aug_mel_1024_reshaped = X_test_aug_mel_1024.reshape(X_test_aug_mel_1024.shape[0], -1)
X_val_aug_mel_1024_reshaped = X_val_aug_mel_1024.reshape(X_val_aug_mel_1024.shape[0], -1)
scaler_aug_mel_1024 = StandardScaler()
X_train_aug_mel_1024_scaled = scaler_aug_mel_1024.fit_transform(X_train_aug_mel_1024_reshaped)
X_test_aug_mel_1024_scaled = scaler_aug_mel_1024.transform(X_test_aug_mel_1024_reshaped)
X_val_aug_mel_1024_scaled = scaler_aug_mel_1024.transform(X_val_aug_mel_1024_reshaped)
X_train_aug_mel_1024 = X_train_aug_mel_1024_scaled.reshape(X_train_aug_mel_1024.shape[0], X_train_aug_mel_1024.shape[1], X_train_aug_mel_1024.shape[2])
X_val_aug_mel_1024 = X_val_aug_mel_1024_scaled.reshape(X_val_aug_mel_1024.shape[0], X_val_aug_mel_1024.shape[1], X_val_aug_mel_1024.shape[2])
X_test_aug_mel_1024 = X_test_aug_mel_1024_scaled.reshape(X_test_aug_mel_1024.shape[0], X_test_aug_mel_1024.shape[1], X_test_aug_mel_1024.shape[2])

# Reshape for 2D-CNN 1024
X_train_aug_mel_1024 = X_train_aug_mel_1024.reshape(X_train_aug_mel_1024.shape[0], X_train_aug_mel_1024.shape[1], X_train_aug_mel_1024.shape[2], 1)
X_val_aug_mel_1024 = X_val_aug_mel_1024.reshape(X_val_aug_mel_1024.shape[0], X_val_aug_mel_1024.shape[1], X_val_aug_mel_1024.shape[2], 1)
X_test_aug_mel_1024 = X_test_aug_mel_1024.reshape(X_test_aug_mel_1024.shape[0], X_test_aug_mel_1024.shape[1], X_test_aug_mel_1024.shape[2], 1)

print(f"Training 1024 shape: {X_train_aug_mel_1024.shape}")
print(f"Validation 1024 shape: {X_val_aug_mel_1024.shape}")
print(f"Test 1024 shape: {X_test_aug_mel_1024.shape}")

# Normalize 2048
X_train_aug_mel_2048_reshaped = X_train_aug_mel_2048.reshape(X_train_aug_mel_2048.shape[0], -1)
X_test_aug_mel_2048_reshaped = X_test_aug_mel_2048.reshape(X_test_aug_mel_2048.shape[0], -1)
X_val_aug_mel_2048_reshaped = X_val_aug_mel_2048.reshape(X_val_aug_mel_2048.shape[0], -1)
scaler_aug_mel_2048 = StandardScaler()
X_train_aug_mel_2048_scaled = scaler_aug_mel_2048.fit_transform(X_train_aug_mel_2048_reshaped)
X_test_aug_mel_2048_scaled = scaler_aug_mel_2048.transform(X_test_aug_mel_2048_reshaped)
X_val_aug_mel_2048_scaled = scaler_aug_mel_2048.transform(X_val_aug_mel_2048_reshaped)
X_train_aug_mel_2048 = X_train_aug_mel_2048_scaled.reshape(X_train_aug_mel_2048.shape[0], X_train_aug_mel_2048.shape[1], X_train_aug_mel_2048.shape[2])
X_val_aug_mel_2048 = X_val_aug_mel_2048_scaled.reshape(X_val_aug_mel_2048.shape[0], X_val_aug_mel_2048.shape[1], X_val_aug_mel_2048.shape[2])
X_test_aug_mel_2048 = X_test_aug_mel_2048_scaled.reshape(X_test_aug_mel_2048.shape[0], X_test_aug_mel_2048.shape[1], X_test_aug_mel_2048.shape[2])

# Reshape for 2D-CNN 2048
X_train_aug_mel_2048 = X_train_aug_mel_2048.reshape(X_train_aug_mel_2048.shape[0], X_train_aug_mel_2048.shape[1], X_train_aug_mel_2048.shape[2], 1)
X_val_aug_mel_2048 = X_val_aug_mel_2048.reshape(X_val_aug_mel_2048.shape[0], X_val_aug_mel_2048.shape[1], X_val_aug_mel_2048.shape[2], 1)
X_test_aug_mel_2048 = X_test_aug_mel_2048.reshape(X_test_aug_mel_2048.shape[0], X_test_aug_mel_2048.shape[1], X_test_aug_mel_2048.shape[2], 1)

print(f"Training 2048 shape: {X_train_aug_mel_2048.shape}")
print(f"Validation 2048 shape: {X_val_aug_mel_2048.shape}")
print(f"Test 2048 shape: {X_test_aug_mel_2048.shape}")


In [ ]:
# Prepare Augmented Data for 2D-CNN

# Normalize

scaler_t = StandardScaler()
X_train_t_scaled = scaler_t.fit_transform(X_train_t)
X_test_t_scaled = scaler_t.transform(X_test_t)
X_val_t_scaled = scaler_t.transform(X_val_t)

# Reshape for 1D-CNN / RNN
X_train_t = X_train_t_scaled.reshape(X_train_t_scaled.shape[0], X_train_t_scaled.shape[1], 1)
X_val_t = X_val_t_scaled.reshape(X_val_t_scaled.shape[0], X_val_t_scaled.shape[1], 1)
X_test_t = X_test_t_scaled.reshape(X_test_t_scaled.shape[0], X_test_t_scaled.shape[1], 1)

print(f"Training shape: {X_train_t.shape}")
print(f"Validation shape: {X_val_t.shape}")
print(f"Test shape: {X_test_t.shape}")

In [ ]:
# Prepare Augmented Data for 2D-CNN

# Normalize

scaler_t_aug = StandardScaler()
X_train_t_aug_scaled = scaler_t_aug.fit_transform(X_train_t_aug)
X_test_t_aug_scaled = scaler_t_aug.transform(X_test_t_aug)
X_val_t_aug_scaled = scaler_t_aug.transform(X_val_t_aug)

X_train_t_aug = X_train_t_aug_scaled
X_val_t_aug = X_val_t_aug_scaled
X_test_t_aug = X_test_t_aug_scaled

print(f"Training shape: {X_train_t_aug.shape}")
print(f"Validation shape: {X_val_t_aug.shape}")
print(f"Test shape: {X_test_t_aug.shape}")

In [ ]:
# Prepare Augmented Data for 2D-CNN

# Normalize
X_train_m_reshaped = X_train_m.reshape(X_train_m.shape[0], -1)
X_test_m_reshaped = X_test_m.reshape(X_test_m.shape[0], -1)
X_val_m_reshaped = X_val_m.reshape(X_val_m.shape[0], -1)
scaler_m = StandardScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_m_reshaped)
X_test_m_scaled = scaler_m.transform(X_test_m_reshaped)
X_val_m_scaled = scaler_m.transform(X_val_m_reshaped)
X_train_m = X_train_m_scaled.reshape(X_train_m.shape[0], X_train_m.shape[1], X_train_m.shape[2])
X_val_m = X_val_m_scaled.reshape(X_val_m.shape[0], X_val_m.shape[1], X_val_m.shape[2])
X_test_m = X_test_m_scaled.reshape(X_test_m.shape[0], X_test_m.shape[1], X_test_m.shape[2])

# Reshape for 1D-CNN / RNN
X_train_m = X_train_m.reshape(X_train_m.shape[0], X_train_m.shape[2], X_train_m.shape[1])
X_val_m = X_val_m.reshape(X_val_m.shape[0], X_val_m.shape[2], X_val_m.shape[1])
X_test_m = X_test_m.reshape(X_test_m.shape[0], X_test_m.shape[2], X_test_m.shape[1])

print(f"Training shape: {X_train_m.shape}")
print(f"Validation shape: {X_val_m.shape}")
print(f"Test shape: {X_test_m.shape}")

In [ ]:
# Prepare Augmented Data for 2D-CNN

# Normalize
X_train_m_aug_reshaped = X_train_m_aug.reshape(X_train_m_aug.shape[0], -1)
X_test_m_aug_reshaped = X_test_m_aug.reshape(X_test_m_aug.shape[0], -1)
X_val_m_aug_reshaped = X_val_m_aug.reshape(X_val_m_aug.shape[0], -1)
scaler_m_aug = StandardScaler()
X_train_m_aug_scaled = scaler_m_aug.fit_transform(X_train_m_aug_reshaped)
X_test_m_aug_scaled = scaler_m_aug.transform(X_test_m_aug_reshaped)
X_val_m_aug_scaled = scaler_m_aug.transform(X_val_m_aug_reshaped)
X_train_m_aug = X_train_m_aug_scaled.reshape(X_train_m_aug.shape[0], X_train_m_aug.shape[1], X_train_m_aug.shape[2])
X_val_m_aug = X_val_m_aug_scaled.reshape(X_val_m_aug.shape[0], X_val_m_aug.shape[1], X_val_m_aug.shape[2])
X_test_m_aug = X_test_m_aug_scaled.reshape(X_test_m_aug.shape[0], X_test_m_aug.shape[1], X_test_m_aug.shape[2])

# Reshape for 1D-CNN / RNN
X_train_m_aug = X_train_m_aug.reshape(X_train_m_aug.shape[0], X_train_m_aug.shape[2], X_train_m_aug.shape[1])
X_val_m_aug = X_val_m_aug.reshape(X_val_m_aug.shape[0], X_val_m_aug.shape[2], X_val_m_aug.shape[1])
X_test_m_aug = X_test_m_aug.reshape(X_test_m_aug.shape[0], X_test_m_aug.shape[2], X_test_m_aug.shape[1])

print(f"Training shape: {X_train_m_aug.shape}")
print(f"Validation shape: {X_val_m_aug.shape}")
print(f"Test shape: {X_test_m_aug.shape}")

# 3. Model & Evaluation

In [ ]:
def create_cnn_lstm_model(input_shape, n_conv=3, batch_normalization=False, rnn_layers = 3, num_classes=10):
    input_layer = Input(shape=input_shape)

    for i in range(0, n_conv):
        x = Conv2D(128, (5, 5), activation='relu', padding='same')(input_layer)
        x = MaxPooling2D((2, 2))(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    shape = x.shape
    x = Reshape((shape[1], shape[2] * shape[3]))(x)
    
    x = LSTM(128, return_sequences=True)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = LSTM(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model

def create_cnn_lstm_model_1d(input_shape, n_conv=3, batch_normalization=False, rnn_layers = 3, num_classes=10):
    input_layer = Input(shape=input_shape)

    for i in range(0, n_conv):
        x = Conv1D(128, 5, activation='relu', padding='same')(input_layer)
        x = MaxPooling1D(2)(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    x = LSTM(128, return_sequences=True)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = LSTM(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model

    
def create_cnn_gru_model(input_shape, n_conv=3, batch_normalization=False, num_classes=10):
    input_layer = Input(shape=input_shape)
    
    for i in range(0, n_conv):
        x = Conv2D(128, (5, 5), activation='relu', padding='same')(input_layer)
        x = MaxPooling2D((2, 2))(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    shape = x.shape
    x = Reshape((shape[1], shape[2] * shape[3]))(x)
    
    x = GRU(128, return_sequences=True)(x)
    x = GRU(128, return_sequences=True)(x)
    x = GRU(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model

def create_cnn_gru_model_1d(input_shape, n_conv=3, batch_normalization=False, rnn_layers = 3, num_classes=10):
    input_layer = Input(shape=input_shape)

    for i in range(0, n_conv):
        x = Conv1D(128, 5, activation='relu', padding='same')(input_layer)
        x = MaxPooling1D(2)(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    x = GRU(128, return_sequences=True)(x)
    x = GRU(128, return_sequences=True)(x)
    x = GRU(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model


def create_cnn_bilstm_model(input_shape, n_conv=3, batch_normalization=False, num_classes=10):
    input_layer = Input(shape=input_shape)
    
    for i in range(0, n_conv):
        x = Conv2D(128, (5, 5), activation='relu', padding='same')(input_layer)
        x = MaxPooling2D((2, 2))(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    shape = x.shape
    x = Reshape((shape[1], shape[2] * shape[3]))(x)
    
    x = LSTM(128, return_sequences=True)(x)
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = LSTM(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model

def create_cnn_bilstm_model_1d(input_shape, n_conv=3, batch_normalization=False, rnn_layers = 3, num_classes=10):
    input_layer = Input(shape=input_shape)

    for i in range(0, n_conv):
        x = Conv1D(128, 5, activation='relu', padding='same')(input_layer)
        x = MaxPooling1D(2)(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    x = LSTM(128, return_sequences=True)(x)
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = LSTM(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model

def create_cnn_bigru_model(input_shape, n_conv=3, batch_normalization=False, num_classes=10):
    input_layer = Input(shape=input_shape)
    
    for i in range(0, n_conv):
        x = Conv2D(128, (5, 5), activation='relu', padding='same')(input_layer)
        x = MaxPooling2D((2, 2))(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    shape = x.shape
    x = Reshape((shape[1], shape[2] * shape[3]))(x)
    
    x = GRU(128, return_sequences=True)(x)
    x = Bidirectional(GRU(128, return_sequences=True))(x)
    x = GRU(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model

def create_cnn_bigru_model_1d(input_shape, n_conv=3, batch_normalization=False, rnn_layers = 3, num_classes=10):
    input_layer = Input(shape=input_shape)

    for i in range(0, n_conv):
        x = Conv1D(128, 5, activation='relu', padding='same')(input_layer)
        x = MaxPooling1D(2)(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        x = Dropout(0.25)(x)
    
    x = GRU(128, return_sequences=True)(x)
    x = Bidirectional(GRU(128, return_sequences=True))(x)
    x = GRU(64, return_sequences=True)(x)
    
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    
    return model

def create_lstm_model(input_shape, batch_normalization=False, num_classes=10):
    input_layer = Input(shape=input_shape)
    x = input_layer
    x = LSTM(128, return_sequences=True)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    if batch_normalization:
        x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model


def create_mlp_model(input_shape, num_classes=10):
    input_layer = Input(shape=input_shape)
    x = input_layer
    x = Dense(64, activation='relu')(x)
    x = Dense(128, activation='relu')(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    output_layer = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return model

def create_attention_mlp_model(input_shape):
    inputs = Input(shape=input_shape)
    
    # --- ATTENTION BLOCK START ---
    # 1. Calculate Attention Weights
    # We compress the info to find global patterns, then expand back to feature size
    # This is similar to a Squeeze-and-Excitation (SE) block
    
    attention_probs = layers.Dense(input_shape[0], activation='softmax', name='attention_vec')(inputs)
    
    # OR, for independent feature gating (often better for tabular):
    # attention_probs = layers.Dense(input_shape[0], activation='sigmoid', name='attention_vec')(inputs)

    # 2. Apply Attention (Re-weight the features)
    # Element-wise multiplication: (Features * Weights)
    attention_mul = layers.Multiply(name='attention_mul')([inputs, attention_probs])
    # --- ATTENTION BLOCK END ---

    # 3. Standard MLP (Now receives the "Highlighted" features)
    x = layers.Dense(256, activation='relu')(attention_mul)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(64, activation='relu')(x)
    
    outputs = layers.Dense(10, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=None, verbose=1):
    early_stopping = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
    
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', 
                              factor=0.2, 
                              patience=5, 
                              min_lr=1e-7,
                              mode='min')
    
    if batch_size is None:
        batch_size = 32
    
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=epochs,
                        batch_size=batch_size,
                        callbacks=[early_stopping, reduce_lr], verbose=verbose)
    return history, model

In [ ]:
def evaluate_model(model, X_test, y_test, target_names):
    test_loss, test_accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")
    y_pred = model.predict(X_test)
    y_pred = np.argmax(y_pred, axis=1)
    y_test = np.argmax(y_test, axis=1)
    print("\nClassification Report")
    print(classification_report(y_test, y_pred, target_names=target_names))
    print("\nConfusion Matrix")
    y_test_labeled = []
    for y in y_test:
        y_test_labeled.append(target_names[y])
    y_pred_labeled = []
    for y in y_pred:
        y_pred_labeled.append(target_names[y])
    cm = confusion_matrix(y_test_labeled, y_pred_labeled)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    return test_accuracy, test_loss

In [ ]:
def plot_history(history):
    plt.figure(figsize=(12, 4))
    
    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(history['loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    
    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['accuracy'], label='Train Accuracy')
    plt.plot(history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    
    plt.show()

In [ ]:
import json
def save_model(model, path):
    os.makedirs(Path(path).parent, exist_ok=True)
    model.save(path)

def save_history(history, path):
    os.makedirs(Path(path).parent, exist_ok=True)
    with open(path, "w") as f:
        json.dump(history, f, indent=4)

In [ ]:
def evaluate_model_song_level(model, X_test_segment, y_test_segment, target_names, feature_type, visualize=True, return_predicted_probs = False):
    if "tzanetakis" in feature_type:
        SEGMENTS_PER_SONG = 10 
    else:
        SEGMENTS_PER_SONG = 19 

    print(f"Evaluating with {SEGMENTS_PER_SONG} segments per song...")

    num_songs = len(X_test_segment) // SEGMENTS_PER_SONG
    total_valid_segments = num_songs * SEGMENTS_PER_SONG
    
    X_truncated = X_test_segment[:total_valid_segments]
    y_truncated = y_test_segment[:total_valid_segments]
    
    if y_truncated.ndim > 1: 
        y_truncated = np.argmax(y_truncated, axis=1)

    y_song_true = []
    y_song_pred_class = []
    y_song_pred_prob = []

    for i in range(num_songs):
        start_idx = i * SEGMENTS_PER_SONG
        end_idx = start_idx + SEGMENTS_PER_SONG
        
        song_segments = X_truncated[start_idx:end_idx]
        
        true_label = y_truncated[start_idx]
        y_song_true.append(true_label)
        
        segment_probs = model.predict(song_segments, verbose=0)
        
        avg_song_prob = np.mean(segment_probs, axis=0)
        y_song_pred_prob.append(avg_song_prob)
        
        y_song_pred_class.append(np.argmax(avg_song_prob))

    y_song_true = np.array(y_song_true)
    y_song_pred_class = np.array(y_song_pred_class)
    y_song_pred_prob = np.array(y_song_pred_prob)

    accuracy = accuracy_score(y_song_true, y_song_pred_class)
    
    y_true_oh = to_categorical(y_song_true, num_classes=len(target_names))
    y_pred_prob_clipped = np.clip(y_song_pred_prob, 1e-15, 1 - 1e-15)
    loss = -np.mean(np.sum(y_true_oh * np.log(y_pred_prob_clipped), axis=1))

    if visualize:
        print(f"Test Loss: {loss:.4f} | Song-Level Accuracy: {accuracy:.4f}")
        print("\nClassification Report")
        print(classification_report(y_song_true, y_song_pred_class, target_names=target_names))
        
        print("\nConfusion Matrix")
        cm = confusion_matrix(y_song_true, y_song_pred_class)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=target_names, yticklabels=target_names)
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.title(f'Confusion Matrix ({feature_type})')
        plt.show()

    if return_predicted_probs:
        return y_song_pred_prob, accuracy, loss
    else:
        return accuracy, loss

In [ ]:
if(os.path.exists(TZANETAKIS_MODEL_PATH + 't_mlp_aug_model.h5')):
    t_mlp_aug_model = load_model(TZANETAKIS_MODEL_PATH + 't_mlp_aug_model.h5')
    with open(TZANETAKIS_HISTORY_PATH + 't_mlp_aug_history.json', 'r') as f:
        t_mlp_aug_model_history = json.load(f)
    print("Loaded existing t_mlp_aug_model.h5 model and history.")
else:
    t_mlp_aug_model = create_mlp_model(X_train_t_aug.shape[1:])
    t_mlp_model_aug_history, t_mlp_aug_model = train_model(t_mlp_aug_model, X_train_t_aug, y_train_t_aug, X_val_t_aug, y_val_t_aug, epochs=50, batch_size=16)
    save_model(t_mlp_aug_model, 'model/t_mlp_aug_model.h5')
    save_history(t_mlp_aug_model_history, 'history/t_mlp_aug_history.json')
    
plot_history(t_mlp_aug_model_history)
print("Segment Level Evaluation")
t_mlp_aug_model_test_acc, t_mlp_aug_model_test_loss = evaluate_model(t_mlp_aug_model, X_test_t_aug, y_test_t_aug, genres)
print("Song Level Evaluation")
t_mlp_aug_model_test_acc_song, t_mlp_aug_model_test_loss_song = evaluate_model_song_level(t_mlp_aug_model,X_test_t_aug, y_test_t_aug, genres, "tzanetakis_vector")

In [ ]:
if(os.path.exists(TZANETAKIS_MODEL_PATH + 't_mlp_model.h5')):
    t_mlp_model = load_model(TZANETAKIS_MODEL_PATH + 't_mlp_model.h5')
    with open(TZANETAKIS_HISTORY_PATH + 't_mlp_history.json', 'r') as f:
        t_mlp_model_history = json.load(f)
    print("Loaded existing t_mlp_model.h5 model and history.")
else:
    t_mlp_model = create_mlp_model(X_train_t.shape[1:])
    t_mlp_model_history, t_mlp_model = train_model(t_mlp_model, X_train_t, y_train_t, X_val_t, y_val_t, epochs=50, batch_size=16)
    save_model(t_mlp_model, 'model/t_mlp_model.h5')
    t_mlp_model_history = t_mlp_model_history.history
    save_history(t_mlp_model_history, 'history/t_mlp_history.json')
    
plot_history(t_mlp_model_history)
t_mlp_model_test_acc, t_mlp_model_test_loss = evaluate_model(t_mlp_model, X_test_t, y_test_t, genres)

In [ ]:
if(os.path.exists(TZANETAKIS_MODEL_PATH + 'm_lstm_aug_model.h5')):
    m_lstm_aug_model = load_model(TZANETAKIS_MODEL_PATH + 'm_lstm_aug_model.h5')
    with open(TZANETAKIS_HISTORY_PATH + 'm_lstm_aug_history.json', 'r') as f:
        m_lstm_aug_model_history = json.load(f)
    print("Loaded existing m_lstm_aug_model.h5 model and history.")
else:
    m_lstm_aug_model = create_lstm_model(X_train_m_aug.shape[1:])
    m_lstm_aug_model_history_obj, m_lstm_aug_model = train_model(
        m_lstm_aug_model,
        X_train_m_aug, y_train_m_aug,
        X_val_m_aug, y_val_m_aug,
        epochs=50, batch_size=16
    )
    m_lstm_aug_model_history = m_lstm_aug_model_history_obj.history
    save_model(m_lstm_aug_model, 'model/m_lstm_aug_model.h5')
    save_history(m_lstm_aug_model_history, 'history/m_lstm_aug_history.json')
    

plot_history(m_lstm_aug_model_history)
print("Segment Level Evaluation")
m_lstm_aug_model_test_acc, m_lstm_aug_model_test_loss = evaluate_model(
    m_lstm_aug_model, X_test_m_aug, y_test_m_aug, genres
)
print("Song Level Evaluation")
m_lstm_aug_model_test_acc_song, m_lstm_aug_model_test_loss_song = evaluate_model_song_level(
    m_lstm_aug_model,
    X_test_m_aug,
    y_test_m_aug,
    genres,
    "tzanetakis_matrix"
)

In [ ]:
if(os.path.exists(TZANETAKIS_MODEL_PATH + 'm_lstm_model.h5')):
    m_lstm_model = load_model(TZANETAKIS_MODEL_PATH + 'm_lstm_model.h5')
    with open(TZANETAKIS_HISTORY_PATH + 'm_lstm_history.json', 'r') as f:
        m_lstm_model_history = json.load(f)
    print("Loaded existing m_lstm_model.h5 model and history.")
else:
    m_lstm_model = create_lstm_model(X_train_m.shape[1:])
    m_lstm_model_history, m_lstm_model = train_model(m_lstm_model, X_train_m, y_train_m, X_val_m, y_val_m, epochs=50, batch_size=16)
    save_model(m_lstm_model, 'model/m_lstm_model.h5')
    save_history(m_lstm_model_history, 'history/m_lstm_history.json')
    
plot_history(m_lstm_model_history)
m_lstm_model_test_acc, m_lstm_model_test_loss = evaluate_model(m_lstm_model, X_test_m, y_test_m, genres)

In [ ]:
if(os.path.exists(MFCC_MODEL_PATH + 'mfcc_cnn_lstm_model.h5')):
    mfcc_cnn_lstm_model = load_model(MFCC_MODEL_PATH + 'mfcc_cnn_lstm_model.h5')
    with open(MFCC_HISTORY_PATH + 'mfcc_cnn_lstm_history.json', 'r') as f:
        mfcc_cnn_lstm_history = json.load(f)
    print("Loaded existing mfcc_cnn_lstm.h5 model and history.")

else:
    mfcc_cnn_lstm_model = create_cnn_lstm_model(X_train_aug_mfcc.shape[1:])
    mfcc_cnn_lstm_history_obj, mfcc_cnn_lstm_model = train_model(
        mfcc_cnn_lstm_model,
        X_train_aug_mfcc, y_train_aug_mfcc,
        X_val_aug_mfcc, y_val_aug_mfcc,
        epochs=50
    )
    mfcc_cnn_lstm_history = mfcc_cnn_lstm_history_obj.history

    save_model(mfcc_cnn_lstm_model, 'model/mfcc_cnn_lstm_model.h5')
    save_history(mfcc_cnn_lstm_history, 'history/mfcc_cnn_lstm_history.json')

plot_history(mfcc_cnn_lstm_history)

print("Segment Level Evaluation")
mfcc_cnn_lstm_test_acc, mfcc_cnn_lstm_test_loss = evaluate_model(
    mfcc_cnn_lstm_model, X_test_aug_mfcc, y_test_aug_mfcc, genres
)

print("Song Level Evaluation")
mfcc_cnn_lstm_test_acc_song, mfcc_cnn_lstm_test_loss_song = evaluate_model_song_level(
    mfcc_cnn_lstm_model,
    X_test_aug_mfcc,
    y_test_aug_mfcc,
    genres,
    "mfcc"
)

In [ ]:
if(os.path.exists(MFCC_MODEL_PATH + 'mfcc_cnn_gru_model.h5')):
    mfcc_cnn_gru_model = load_model(MFCC_MODEL_PATH + 'mfcc_cnn_gru_model.h5')
    with open(MFCC_HISTORY_PATH + 'mfcc_cnn_gru_history.json', 'r') as f:
        mfcc_cnn_gru_history = json.load(f)
    print("Loaded existing mfcc_cnn_gru.h5 model and history.")

else:
    mfcc_cnn_gru_model = create_cnn_gru_model(X_train_aug_mfcc.shape[1:])
    mfcc_cnn_gru_history_obj, mfcc_cnn_gru_model = train_model(
        mfcc_cnn_gru_model,
        X_train_aug_mfcc, y_train_aug_mfcc,
        X_val_aug_mfcc, y_val_aug_mfcc,
        epochs=50
    )
    mfcc_cnn_gru_history = mfcc_cnn_gru_history_obj.history

    save_model(mfcc_cnn_gru_model, 'model/mfcc_cnn_gru_model.h5')
    save_history(mfcc_cnn_gru_history, 'history/mfcc_cnn_gru_history.json')

plot_history(mfcc_cnn_gru_history)

print("Segment Level Evaluation")
mfcc_cnn_gru_test_acc, mfcc_cnn_gru_test_loss = evaluate_model(
    mfcc_cnn_gru_model, X_test_aug_mfcc, y_test_aug_mfcc, genres
)

print("Song Level Evaluation")
mfcc_cnn_gru_test_acc_song, mfcc_cnn_gru_test_loss_song = evaluate_model_song_level(
    mfcc_cnn_gru_model,
    X_test_aug_mfcc,
    y_test_aug_mfcc,
    genres,
    "mfcc"
)

In [ ]:
if(os.path.exists(MFCC_MODEL_PATH + 'mfcc_cnn_bilstm_model.h5')):
    mfcc_cnn_bilstm_model = load_model(MFCC_MODEL_PATH + 'mfcc_cnn_bilstm_model.h5')
    with open(MFCC_HISTORY_PATH + 'mfcc_cnn_bilstm_history.json', 'r') as f:
        mfcc_cnn_bilstm_history = json.load(f)
    print("Loaded existing mfcc_cnn_bilstm.h5 model and history.")

else:
    mfcc_cnn_bilstm_model = create_cnn_bilstm_model(X_train_aug_mfcc.shape[1:])
    mfcc_cnn_bilstm_history_obj, mfcc_cnn_bilstm_model = train_model(
        mfcc_cnn_bilstm_model,
        X_train_aug_mfcc, y_train_aug_mfcc,
        X_val_aug_mfcc, y_val_aug_mfcc,
        epochs=50
    )
    mfcc_cnn_bilstm_history = mfcc_cnn_bilstm_history_obj.history

    save_model(mfcc_cnn_bilstm_model, 'model/mfcc_cnn_bilstm_model.h5')
    save_history(mfcc_cnn_bilstm_history, 'history/mfcc_cnn_bilstm_history.json')

plot_history(mfcc_cnn_bilstm_history)

print("Segment Level Evaluation")
mfcc_cnn_bilstm_test_acc, mfcc_cnn_bilstm_test_loss = evaluate_model(
    mfcc_cnn_bilstm_model, X_test_aug_mfcc, y_test_aug_mfcc, genres
)

print("Song Level Evaluation")
mfcc_cnn_bilstm_test_acc_song, mfcc_cnn_bilstm_test_loss_song = evaluate_model_song_level(
    mfcc_cnn_bilstm_model,
    X_test_aug_mfcc,
    y_test_aug_mfcc,
    genres,
    "mfcc"
)

In [ ]:
if(os.path.exists(MFCC_MODEL_PATH + 'mfcc_cnn_bigru_model.h5')):
    mfcc_cnn_bigru_model = load_model(MFCC_MODEL_PATH + 'mfcc_cnn_bigru_model.h5')
    with open(MFCC_HISTORY_PATH + 'mfcc_cnn_bigru_history.json', 'r') as f:
        mfcc_cnn_bigru_history = json.load(f)
    print("Loaded existing mfcc_cnn_bigru.h5 model and history.")

else:
    mfcc_cnn_bigru_model = create_cnn_bigru_model(X_train_aug_mfcc.shape[1:])
    mfcc_cnn_bigru_history_obj, mfcc_cnn_bigru_model = train_model(
        mfcc_cnn_bigru_model,
        X_train_aug_mfcc, y_train_aug_mfcc,
        X_val_aug_mfcc, y_val_aug_mfcc,
        epochs=50
    )
    mfcc_cnn_bigru_history = mfcc_cnn_bigru_history_obj.history

    save_model(mfcc_cnn_bigru_model, 'model/mfcc_cnn_bigru_model.h5')
    save_history(mfcc_cnn_bigru_history, 'history/mfcc_cnn_bigru_history.json')

plot_history(mfcc_cnn_bigru_history)

print("Segment Level Evaluation")
mfcc_cnn_bigru_test_acc, mfcc_cnn_bigru_test_loss = evaluate_model(
    mfcc_cnn_bigru_model, X_test_aug_mfcc, y_test_aug_mfcc, genres
)

print("Song Level Evaluation")
mfcc_cnn_bigru_test_acc_song, mfcc_cnn_bigru_test_loss_song = evaluate_model_song_level(
    mfcc_cnn_bigru_model,
    X_test_aug_mfcc,
    y_test_aug_mfcc,
    genres,
    "mfcc"
)

In [ ]:
if(os.path.exists(MEL_1024_MODEL_PATH + 'mel_1024_cnn_lstm_model.h5')):
    mel_1024_cnn_lstm_model = load_model(MEL_1024_MODEL_PATH + 'mel_1024_cnn_lstm_model.h5')
    with open(MEL_1024_HISTORY_PATH + 'mel_1024_cnn_lstm_history.json', 'r') as f:
        mel_1024_cnn_lstm_history = json.load(f)
    print("Loaded existing mel_1024_cnn_lstm.h5 model and history.")

else:
    mel_1024_cnn_lstm_model = create_cnn_lstm_model(X_train_aug_mel_1024.shape[1:])
    mel_1024_cnn_lstm_history_obj, mel_1024_cnn_lstm_model = train_model(
        mel_1024_cnn_lstm_model,
        X_train_aug_mel_1024, y_train_aug_mel_1024,
        X_val_aug_mel_1024, y_val_aug_mel_1024,
        epochs=50
    )
    mel_1024_cnn_lstm_history = mel_1024_cnn_lstm_history_obj.history

    save_model(mel_1024_cnn_lstm_model, 'model/mel_1024_cnn_lstm_model.h5')
    save_history(mel_1024_cnn_lstm_history, 'history/mel_1024_cnn_lstm_history.json')

plot_history(mel_1024_cnn_lstm_history)

print("Segment Level Evaluation")
mel_1024_cnn_lstm_test_acc, mel_1024_cnn_lstm_test_loss = evaluate_model(
    mel_1024_cnn_lstm_model, X_test_aug_mel_1024, y_test_aug_mel_1024, genres
)

print("Song Level Evaluation")
mel_1024_cnn_lstm_test_acc_song, mel_1024_cnn_lstm_test_loss_song = evaluate_model_song_level(
    mel_1024_cnn_lstm_model,
    X_test_aug_mel_1024,
    y_test_aug_mel_1024,
    genres,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_1024_MODEL_PATH + 'mel_1024_cnn_gru_model.h5')):
    mel_1024_cnn_gru_model = load_model(MEL_1024_MODEL_PATH + 'mel_1024_cnn_gru_model.h5')
    with open(MEL_1024_HISTORY_PATH + 'mel_1024_cnn_gru_history.json', 'r') as f:
        mel_1024_cnn_gru_history = json.load(f)
    print("Loaded existing mel_1024_cnn_gru.h5 model and history.")

else:
    mel_1024_cnn_gru_model = create_cnn_gru_model(X_train_aug_mel_1024.shape[1:])
    mel_1024_cnn_gru_history_obj, mel_1024_cnn_gru_model = train_model(
        mel_1024_cnn_gru_model,
        X_train_aug_mel_1024, y_train_aug_mel_1024,
        X_val_aug_mel_1024, y_val_aug_mel_1024,
        epochs=50
    )
    mel_1024_cnn_gru_history = mel_1024_cnn_gru_history_obj.history

    save_model(mel_1024_cnn_gru_model, 'model/mel_1024_cnn_gru_model.h5')
    save_history(mel_1024_cnn_gru_history, 'history/mel_1024_cnn_gru_history.json')

plot_history(mel_1024_cnn_gru_history)

print("Segment Level Evaluation")
mel_1024_cnn_gru_test_acc, mel_1024_cnn_gru_test_loss = evaluate_model(
    mel_1024_cnn_gru_model, X_test_aug_mel_1024, y_test_aug_mel_1024, genres
)

print("Song Level Evaluation")
mel_1024_cnn_gru_test_acc_song, mel_1024_cnn_gru_test_loss_song = evaluate_model_song_level(
    mel_1024_cnn_gru_model,
    X_test_aug_mel_1024,
    y_test_aug_mel_1024,
    genres,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_1024_MODEL_PATH + 'mel_1024_cnn_bilstm_model.h5')):
    mel_1024_cnn_bilstm_model = load_model(MEL_1024_MODEL_PATH + 'mel_1024_cnn_bilstm_model.h5')
    with open(MEL_1024_HISTORY_PATH + 'mel_1024_cnn_bilstm_history.json', 'r') as f:
        mel_1024_cnn_bilstm_history = json.load(f)
    print("Loaded existing mel_1024_cnn_bilstm.h5 model and history.")

else:
    mel_1024_cnn_bilstm_model = create_cnn_bilstm_model(X_train_aug_mel_1024.shape[1:])
    mel_1024_cnn_bilstm_history_obj, mel_1024_cnn_bilstm_model = train_model(
        mel_1024_cnn_bilstm_model,
        X_train_aug_mel_1024, y_train_aug_mel_1024,
        X_val_aug_mel_1024, y_val_aug_mel_1024,
        epochs=50
    )
    mel_1024_cnn_bilstm_history = mel_1024_cnn_bilstm_history_obj.history

    save_model(mel_1024_cnn_bilstm_model, 'model/mel_1024_cnn_bilstm_model.h5')
    save_history(mel_1024_cnn_bilstm_history, 'history/mel_1024_cnn_bilstm_history.json')

plot_history(mel_1024_cnn_bilstm_history)

print("Segment Level Evaluation")
mel_1024_cnn_bilstm_test_acc, mel_1024_cnn_bilstm_test_loss = evaluate_model(
    mel_1024_cnn_bilstm_model, X_test_aug_mel_1024, y_test_aug_mel_1024, genres
)

print("Song Level Evaluation")
mel_1024_cnn_bilstm_test_acc_song, mel_1024_cnn_bilstm_test_loss_song = evaluate_model_song_level(
    mel_1024_cnn_bilstm_model,
    X_test_aug_mel_1024,
    y_test_aug_mel_1024,
    genres,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_1024_MODEL_PATH + 'mel_1024_cnn_bigru_model.h5')):
    mel_1024_cnn_bigru_model = load_model(MEL_1024_MODEL_PATH + 'mel_1024_cnn_bigru_model.h5')
    with open(MEL_1024_HISTORY_PATH + 'mel_1024_cnn_bigru_history.json', 'r') as f:
        mel_1024_cnn_bigru_history = json.load(f)
    print("Loaded existing mel_1024_cnn_bigru.h5 model and history.")

else:
    mel_1024_cnn_bigru_model = create_cnn_bigru_model(X_train_aug_mel_1024.shape[1:])
    mel_1024_cnn_bigru_history_obj, mel_1024_cnn_bigru_model = train_model(
        mel_1024_cnn_bigru_model,
        X_train_aug_mel_1024, y_train_aug_mel_1024,
        X_val_aug_mel_1024, y_val_aug_mel_1024,
        epochs=50
    )
    mel_1024_cnn_bigru_history = mel_1024_cnn_bigru_history_obj.history

    save_model(mel_1024_cnn_bigru_model, 'model/mel_1024_cnn_bigru_model.h5')
    save_history(mel_1024_cnn_bigru_history, 'history/mel_1024_cnn_bigru_history.json')

plot_history(mel_1024_cnn_bigru_history)

print("Segment Level Evaluation")
mel_1024_cnn_bigru_test_acc, mel_1024_cnn_bigru_test_loss = evaluate_model(
    mel_1024_cnn_bigru_model, X_test_aug_mel_1024, y_test_aug_mel_1024, genres
)

print("Song Level Evaluation")
mel_1024_cnn_bigru_test_acc_song, mel_1024_cnn_bigru_test_loss_song = evaluate_model_song_level(
    mel_1024_cnn_bigru_model,
    X_test_aug_mel_1024,
    y_test_aug_mel_1024,
    genres,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_2048_MODEL_PATH + 'mel_2048_cnn_lstm_model.h5')):
    mel_2048_cnn_lstm_model = load_model(MEL_2048_MODEL_PATH + 'mel_2048_cnn_lstm_model.h5')
    with open(MEL_2048_HISTORY_PATH + 'mel_2048_cnn_lstm_history.json', 'r') as f:
        mel_2048_cnn_lstm_history = json.load(f)
    print("Loaded existing mel_2048_cnn_lstm.h5 model and history.")

else:
    mel_2048_cnn_lstm_model = create_cnn_lstm_model(X_train_aug_mel_2048.shape[1:])
    mel_2048_cnn_lstm_history_obj, mel_2048_cnn_lstm_model = train_model(
        mel_2048_cnn_lstm_model,
        X_train_aug_mel_2048, y_train_aug_mel_2048,
        X_val_aug_mel_2048, y_val_aug_mel_2048,
        epochs=50
    )
    mel_2048_cnn_lstm_history = mel_2048_cnn_lstm_history_obj.history

    save_model(mel_2048_cnn_lstm_model, 'model/mel_2048_cnn_lstm_model.h5')
    save_history(mel_2048_cnn_lstm_history, 'history/mel_2048_cnn_lstm_history.json')

plot_history(mel_2048_cnn_lstm_history)

print("Segment Level Evaluation")
mel_2048_cnn_lstm_test_acc, mel_2048_cnn_lstm_test_loss = evaluate_model(
    mel_2048_cnn_lstm_model, X_test_aug_mel_2048, y_test_aug_mel_2048, genres
)

print("Song Level Evaluation")
mel_2048_cnn_lstm_test_acc_song, mel_2048_cnn_lstm_test_loss_song = evaluate_model_song_level(
    mel_2048_cnn_lstm_model,
    X_test_aug_mel_2048,
    y_test_aug_mel_2048,
    genres,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_2048_MODEL_PATH + 'mel_2048_cnn_gru_model.h5')):
    mel_2048_cnn_gru_model = load_model(MEL_2048_MODEL_PATH + 'mel_2048_cnn_gru_model.h5')
    with open(MEL_2048_HISTORY_PATH + 'mel_2048_cnn_gru_history.json', 'r') as f:
        mel_2048_cnn_gru_history = json.load(f)
    print("Loaded existing mel_2048_cnn_gru.h5 model and history.")

else:
    mel_2048_cnn_gru_model = create_cnn_gru_model(X_train_aug_mel_2048.shape[1:])
    mel_2048_cnn_gru_history_obj, mel_2048_cnn_gru_model = train_model(
        mel_2048_cnn_gru_model,
        X_train_aug_mel_2048, y_train_aug_mel_2048,
        X_val_aug_mel_2048, y_val_aug_mel_2048,
        epochs=50
    )
    mel_2048_cnn_gru_history = mel_2048_cnn_gru_history_obj.history

    save_model(mel_2048_cnn_gru_model, 'model/mel_2048_cnn_gru_model.h5')
    save_history(mel_2048_cnn_gru_history, 'history/mel_2048_cnn_gru_history.json')

plot_history(mel_2048_cnn_gru_history)

print("Segment Level Evaluation")
mel_2048_cnn_gru_test_acc, mel_2048_cnn_gru_test_loss = evaluate_model(
    mel_2048_cnn_gru_model, X_test_aug_mel_2048, y_test_aug_mel_2048, genres
)

print("Song Level Evaluation")
mel_2048_cnn_gru_test_acc_song, mel_2048_cnn_gru_test_loss_song = evaluate_model_song_level(
    mel_2048_cnn_gru_model,
    X_test_aug_mel_2048,
    y_test_aug_mel_2048,
    genres,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_2048_MODEL_PATH + 'mel_2048_cnn_bilstm_model.h5')):
    mel_2048_cnn_bilstm_model = load_model(MEL_2048_MODEL_PATH + 'mel_2048_cnn_bilstm_model.h5')
    with open(MEL_2048_HISTORY_PATH + 'mel_2048_cnn_bilstm_history.json', 'r') as f:
        mel_2048_cnn_bilstm_history = json.load(f)
    print("Loaded existing mel_2048_cnn_bilstm.h5 model and history.")

else:
    mel_2048_cnn_bilstm_model = create_cnn_bilstm_model(X_train_aug_mel_2048.shape[1:])
    mel_2048_cnn_bilstm_history_obj, mel_2048_cnn_bilstm_model = train_model(
        mel_2048_cnn_bilstm_model,
        X_train_aug_mel_2048, y_train_aug_mel_2048,
        X_val_aug_mel_2048, y_val_aug_mel_2048,
        epochs=50
    )
    mel_2048_cnn_bilstm_history = mel_2048_cnn_bilstm_history_obj.history

    save_model(mel_2048_cnn_bilstm_model, 'model/mel_2048_cnn_bilstm_model.h5')
    save_history(mel_2048_cnn_bilstm_history, 'history/mel_2048_cnn_bilstm_history.json')

plot_history(mel_2048_cnn_bilstm_history)

print("Segment Level Evaluation")
mel_2048_cnn_bilstm_test_acc, mel_2048_cnn_bilstm_test_loss = evaluate_model(
    mel_2048_cnn_bilstm_model, X_test_aug_mel_2048, y_test_aug_mel_2048, genres
)

print("Song Level Evaluation")
mel_2048_cnn_bilstm_test_acc_song, mel_2048_cnn_bilstm_test_loss_song = evaluate_model_song_level(
    mel_2048_cnn_bilstm_model,
    X_test_aug_mel_2048,
    y_test_aug_mel_2048,
    genres,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_2048_MODEL_PATH + 'mel_2048_cnn_bigru_model.h5')):
    mel_2048_cnn_bigru_model = load_model(MEL_2048_MODEL_PATH + 'mel_2048_cnn_bigru_model.h5')
    with open(MEL_2048_HISTORY_PATH + 'mel_2048_cnn_bigru_history.json', 'r') as f:
        mel_2048_cnn_bigru_history = json.load(f)
    print("Loaded existing mel_2048_cnn_bigru.h5 model and history.")

else:
    mel_2048_cnn_bigru_model = create_cnn_bigru_model(X_train_aug_mel_2048.shape[1:])
    mel_2048_cnn_bigru_history_obj, mel_2048_cnn_bigru_model = train_model(
        mel_2048_cnn_bigru_model,
        X_train_aug_mel_2048, y_train_aug_mel_2048,
        X_val_aug_mel_2048, y_val_aug_mel_2048,
        epochs=50
    )
    mel_2048_cnn_bigru_history = mel_2048_cnn_bigru_history_obj.history

    save_model(mel_2048_cnn_bigru_model, 'model/mel_2048_cnn_bigru_model.h5')
    save_history(mel_2048_cnn_bigru_history, 'history/mel_2048_cnn_bigru_history.json')

plot_history(mel_2048_cnn_bigru_history)

print("Segment Level Evaluation")
mel_2048_cnn_bigru_test_acc, mel_2048_cnn_bigru_test_loss = evaluate_model(
    mel_2048_cnn_bigru_model, X_test_aug_mel_2048, y_test_aug_mel_2048, genres
)

print("Song Level Evaluation")
mel_2048_cnn_bigru_test_acc_song, mel_2048_cnn_bigru_test_loss_song = evaluate_model_song_level(
    mel_2048_cnn_bigru_model,
    X_test_aug_mel_2048,
    y_test_aug_mel_2048,
    genres,
    "mel"
)

In [ ]:
test_acc_df = pd.DataFrame({
    'features': ['mfcc', 'mfcc', 'mfcc', 'mfcc', 'mel_spectrograms_1024', 'mel_spectrograms_1024', 'mel_spectrograms_1024', 'mel_spectrograms_1024', 'mel_spectrograms_2048', 'mel_spectrograms_2048', 'mel_spectrograms_2048', 'mel_spectrograms_2048'],
    "model": ['cnn_lstm', 'cnn_gru', 'cnn_bigru', 'cnn_bilstm', 'cnn_lstm', 'cnn_gru', 'cnn_bigru', 'cnn_bilstm', 'cnn_lstm', 'cnn_gru', 'cnn_bigru', 'cnn_bilstm'],
    'test_acc': [mfcc_cnn_lstm_test_acc, mfcc_cnn_gru_test_acc, mfcc_cnn_bigru_test_acc, mfcc_cnn_bilstm_test_acc, mel_1024_cnn_lstm_test_acc, mel_1024_cnn_gru_test_acc, mel_1024_cnn_bigru_test_acc, mel_2048_cnn_bilstm_test_acc, mel_2048_cnn_lstm_test_acc, mel_2048_cnn_gru_test_acc, mel_2048_cnn_bigru_test_acc, mel_2048_cnn_bilstm_test_acc]
})

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=test_acc_df, x='model', y='test_acc', hue='features', palette='viridis')

ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Model')
ax.set_title('Test Accuracy by Feature & Model', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.bar_label(ax.containers[0], fmt='%.3f', fontsize=9, padding=3)
ax.bar_label(ax.containers[1], fmt='%.3f', fontsize=9, padding=3)
ax.bar_label(ax.containers[2], fmt='%.3f', fontsize=9, padding=3)
plt.tight_layout()

print("🏆 BEST MODEL:")
print("Features:", test_acc_df.sort_values(by="test_acc", ascending=False)['features'].iloc[0])
print("Model:", test_acc_df.sort_values(by="test_acc", ascending=False)['model'].iloc[0])
print("Test Accuracy:", test_acc_df.sort_values(by="test_acc", ascending=False)['test_acc'].iloc[0])

In [ ]:
test_acc_df = pd.DataFrame({
    'features': ['mfcc','mfcc','mfcc','mfcc','mel_1024','mel_1024','mel_1024','mel_1024','mel_2048','mel_2048','mel_2048','mel_2048','mfcc','mfcc','mfcc','mfcc','mel_1024','mel_1024','mel_1024','mel_1024','mel_2048','mel_2048','mel_2048','mel_2048'],
    'model': ['cnn_lstm','cnn_gru','cnn_bigru','cnn_bilstm','cnn_lstm','cnn_gru','cnn_bigru','cnn_bilstm','cnn_lstm','cnn_gru','cnn_bigru','cnn_bilstm','cnn_lstm','cnn_gru','cnn_bigru','cnn_bilstm','cnn_lstm','cnn_gru','cnn_bigru','cnn_bilstm','cnn_lstm','cnn_gru','cnn_bigru','cnn_bilstm'],
    'evaluation_level': ['segment','segment','segment','segment','segment','segment','segment','segment','segment','segment','segment','segment','song','song','song','song','song','song','song','song','song','song','song','song'],
    'test_acc': [
        mfcc_cnn_lstm_test_acc, mfcc_cnn_gru_test_acc, mfcc_cnn_bigru_test_acc, mfcc_cnn_bilstm_test_acc,
        mel_1024_cnn_lstm_test_acc, mel_1024_cnn_gru_test_acc, mel_1024_cnn_bigru_test_acc, mel_1024_cnn_bilstm_test_acc,
        mel_2048_cnn_lstm_test_acc, mel_2048_cnn_gru_test_acc, mel_2048_cnn_bigru_test_acc, mel_2048_cnn_bilstm_test_acc,
        mfcc_cnn_lstm_test_acc_song, mfcc_cnn_gru_test_acc_song, mfcc_cnn_bigru_test_acc_song, mfcc_cnn_bilstm_test_acc_song,
        mel_1024_cnn_lstm_test_acc_song, mel_1024_cnn_gru_test_acc_song, mel_1024_cnn_bigru_test_acc_song, mel_1024_cnn_bilstm_test_acc_song,
        mel_2048_cnn_lstm_test_acc_song, mel_2048_cnn_gru_test_acc_song, mel_2048_cnn_bigru_test_acc_song, mel_2048_cnn_bilstm_test_acc_song
    ]
})

test_acc_df['features_model'] = test_acc_df['features'] + "_" + test_acc_df['model']

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=test_acc_df, x='features_model', y='test_acc', hue='evaluation_level', palette='viridis')

ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Model')
ax.set_title('Test Accuracy by Feature & Model', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.bar_label(ax.containers[0], fmt='%.3f', fontsize=9, padding=3)
ax.bar_label(ax.containers[1], fmt='%.3f', fontsize=9, padding=3)
plt.tight_layout()

print("🏆 BEST MODEL:")
print("Features:", test_acc_df.sort_values(by="test_acc", ascending=False)['features'].iloc[0])
print("Model:", test_acc_df.sort_values(by="test_acc", ascending=False)['model'].iloc[0])
print("Test Accuracy (Song):", test_acc_df.sort_values(by="test_acc", ascending=False)['test_acc'].iloc[0])

# Hyperparameter Tuning & Stacking Ensemble

In [ ]:
import optuna
from tensorflow.keras import backend as K

def create_mlp_optimized(trial, input_shape):
    input_layer = Input(shape=input_shape)
    x = input_layer

    dense_unit_1 = trial.suggest_categorical('dense_unit_1', [64, 128, 256])
    dense_unit_2 = trial.suggest_categorical('dense_unit_2', [64, 128, 256])
    dense_unit_3 = trial.suggest_categorical('dense_unit_3', [64, 128, 256])

    activation_1 = trial.suggest_categorical('activation_1', ['relu', 'sigmoid', 'tanh'])
    activation_2 = trial.suggest_categorical('activation_2', ['relu', 'sigmoid', 'tanh'])
    activation_3 = trial.suggest_categorical('activation_3', ['relu', 'sigmoid', 'tanh'])
    
    x = Dense(dense_unit_1, activation=activation_1)(x)
    x = Dense(dense_unit_2, activation=activation_2)(x)
    x = Dense(dense_unit_3, activation=activation_3)(x)

    use_bn = trial.suggest_categorical('use_bn', [True, False])
    if use_bn:
        x = BatchNormalization()(x)

    dropout_prob = trial.suggest_float('dropout_prob', 0.2, 0.5)
    x = Dropout(dropout_prob)(x)
    
    output_layer = Dense(10, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output_layer)

    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'rmsprop', 'sgd'])
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    
    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    else:
        optimizer = SGD(learning_rate=learning_rate, momentum=0.9)
    
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model
    

def objective_mlp(trial):
    """Objective function for MLP optimization"""
    K.clear_session()
    
    model = create_mlp_optimized(trial, X_train_t_aug.shape[1:])
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    
    early_stop = EarlyStopping(monitor='val_accuracy', patience=10, 
                               restore_best_weights=True, mode='max')
    
    history = model.fit(
        X_train_t_aug, y_train_t_aug,
        validation_data=(X_val_t_aug, y_val_t_aug),
        epochs=50,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )
    
    return max(history.history['val_accuracy'])

# Run Optuna study for CRNN
print("\nStarting Optuna Hyperparameter Search for MLP...")
print("="*60)

study_mlp = optuna.create_study(
    direction='maximize',
    study_name='MLP_Optimization',
    pruner=optuna.pruners.MedianPruner()
)

study_mlp.optimize(objective_mlp, n_trials=25)

print("\n" + "="*60)
print("MLP OPTIMIZATION COMPLETE!")
print("="*60)
print(f"Best Trial Number: {study_mlp.best_trial.number}")
print(f"Best Validation Accuracy: {study_mlp.best_value*100:.2f}%")
print("\nBest Hyperparameters:")
for key, value in study_mlp.best_params.items():
    print(f"  {key:20s}: {value}")
print("="*60)

In [ ]:
def create_mlp_optimized_version(best_params, input_shape):
    input_layer = Input(shape=input_shape)
    x = input_layer
    
    x = Dense(best_params['dense_unit_1'], activation=best_params['activation_1'])(x)
    x = Dense(best_params['dense_unit_2'], activation=best_params['activation_2'])(x)
    x = Dense(best_params['dense_unit_3'], activation=best_params['activation_3'])(x)

    use_bn = best_params['use_bn']
    if use_bn:
        x = BatchNormalization()(x)

    dropout_prob = best_params['dropout_prob']
    x = Dropout(dropout_prob)(x)
    
    output_layer = Dense(10, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output_layer)

    optimizer_name = best_params['optimizer']
    learning_rate = best_params['learning_rate']
    
    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    else:
        optimizer = SGD(learning_rate=learning_rate, momentum=0.9)
    
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
if(os.path.exists(MEL_1024_MODEL_PATH + 't_mlp_aug_optimzied_model.h5')):
    t_mlp_aug_optimzied_model = load_model(MEL_1024_MODEL_PATH + 't_mlp_aug_optimzied_model.h5')
    with open(MEL_1024_HISTORY_PATH + 't_mlp_aug_optimzied_history.json', 'r') as f:
        t_mlp_aug_optimzied_history = json.load(f)
    print("Loaded existing t_mlp_aug_optimzied_model.h5 model and history.")

else:
    t_mlp_aug_optimzied_model = create_mlp_optimized_version(study_mlp.best_params, X_train_t_aug.shape[1:])
    t_mlp_aug_optimzied_history_obj, t_mlp_aug_optimzied_model = train_model(
        t_mlp_aug_optimzied_model,
        X_train_t_aug, y_train_t_aug,
        X_val_t_aug, y_val_t_aug,
        epochs=50, batch_size = study_mlp.best_params['batch_size']
    )
    t_mlp_aug_optimzied_history = t_mlp_aug_optimzied_history_obj.history

    save_model(t_mlp_aug_optimzied_model, 'model/t_mlp_aug_optimzied_model.h5')
    save_history(t_mlp_aug_optimzied_history, 'history/t_mlp_aug_optimzied_history.json')

plot_history(t_mlp_aug_optimzied_history)

print("Segment Level Evaluation")
t_mlp_aug_optimzied_test_acc, t_mlp_aug_optimzied_test_loss = evaluate_model(
    t_mlp_aug_optimzied_model, X_test_t_aug, y_test_t_aug, genres
)

print("Song Level Evaluation")
t_mlp_aug_optimzied_test_acc_song, t_mlp_aug_optimzied_test_loss_song = evaluate_model_song_level(
    t_mlp_aug_optimzied_model, X_test_t_aug, y_test_t_aug, genres,
    "tzanetakis_vector"
)

In [ ]:
# Stacking Ensemble

n_folds = 5
ensemble_models_name = ["mel_2048_cnn_gru"
                        , "mel_1024_cnn_bilstm", 
                        # ,"t_aug_mlp"
                        , "mel_2048_cnn_lstm"
                       ]

num_train_songs = len(X_train_aug_mel_1024_combined) // 19
num_test_songs = len(X_test_aug_mel_1024_combined) // 19

groups_mel = np.repeat(np.arange(num_train_songs), 19)
groups_tz  = np.repeat(np.arange(num_train_songs), 10)

sgkf = StratifiedGroupKFold(n_splits=n_folds)

oof_predictions = np.zeros((num_train_songs, len(ensemble_models_name) * 10))
test_predictions = np.zeros((num_test_songs, len(ensemble_models_name) * 10, n_folds))

model_metrics = {name: {'acc': [], 'loss': []} for name in ensemble_models_name}

n_classes=10

stratity_y = np.argmax(y_train_aug_mel_1024_combined, axis=1)

for fold, (train_index, test_index) in enumerate(sgkf.split(X_train_aug_mel_1024_combined, stratity_y, groups=groups_mel)):
    train_song_ids = np.unique(groups_mel[train_index])
    val_song_ids = np.unique(groups_mel[test_index])
    
    # 1024
    X_train_fold_mel_1024 = X_train_aug_mel_1024_combined[train_index]
    y_train_fold_mel_1024 = y_train_aug_mel_1024_combined[train_index]
    X_val_fold_mel_1024 = X_train_aug_mel_1024_combined[test_index]
    y_val_fold_mel_1024 = y_train_aug_mel_1024_combined[test_index]
    X_test_fold_mel_1024 = X_test_aug_mel_1024_combined
    
    X_train_fold_mel_1024_r = X_train_fold_mel_1024.reshape(X_train_fold_mel_1024.shape[0], -1)
    X_val_fold_mel_1024_r = X_val_fold_mel_1024.reshape(X_val_fold_mel_1024.shape[0], -1)
    X_test_fold_mel_1024_r = X_test_fold_mel_1024.reshape(X_test_fold_mel_1024.shape[0], -1)
    
    scaler_fold_mel_1024 = StandardScaler()
    X_train_fold_mel_1024_s = scaler_fold_mel_1024.fit_transform(X_train_fold_mel_1024_r)
    X_val_fold_mel_1024_s = scaler_fold_mel_1024.transform(X_val_fold_mel_1024_r)
    X_test_fold_mel_1024_s = scaler_fold_mel_1024.transform(X_test_fold_mel_1024_r)
    
    X_train_fold_mel_1024 = X_train_fold_mel_1024_s.reshape(
        X_train_fold_mel_1024.shape[0],
        X_train_fold_mel_1024.shape[1],
        X_train_fold_mel_1024.shape[2], 1
    )
    X_val_fold_mel_1024 = X_val_fold_mel_1024_s.reshape(
        X_val_fold_mel_1024.shape[0],
        X_val_fold_mel_1024.shape[1],
        X_val_fold_mel_1024.shape[2], 1
    )
    X_test_fold_mel_1024 = X_test_fold_mel_1024_s.reshape(
        X_test_fold_mel_1024.shape[0],
        X_test_fold_mel_1024.shape[1],
        X_test_fold_mel_1024.shape[2], 1
    )
    
    # 2048
    X_train_fold_mel_2048 = X_train_aug_mel_2048_combined[train_index]
    y_train_fold_mel_2048 = y_train_aug_mel_2048_combined[train_index]
    X_val_fold_mel_2048 = X_train_aug_mel_2048_combined[test_index]
    y_val_fold_mel_2048 = y_train_aug_mel_2048_combined[test_index]
    X_test_fold_mel_2048 = X_test_aug_mel_2048_combined
    
    X_train_fold_mel_2048_r = X_train_fold_mel_2048.reshape(X_train_fold_mel_2048.shape[0], -1)
    X_val_fold_mel_2048_r = X_val_fold_mel_2048.reshape(X_val_fold_mel_2048.shape[0], -1)
    X_test_fold_mel_2048_r = X_test_fold_mel_2048.reshape(X_test_fold_mel_2048.shape[0], -1)
    
    scaler_fold_mel_2048 = StandardScaler()
    X_train_fold_mel_2048_s = scaler_fold_mel_2048.fit_transform(X_train_fold_mel_2048_r)
    X_val_fold_mel_2048_s = scaler_fold_mel_2048.transform(X_val_fold_mel_2048_r)
    X_test_fold_mel_2048_s = scaler_fold_mel_2048.transform(X_test_fold_mel_2048_r)
    
    X_train_fold_mel_2048 = X_train_fold_mel_2048_s.reshape(
        X_train_fold_mel_2048.shape[0],
        X_train_fold_mel_2048.shape[1],
        X_train_fold_mel_2048.shape[2], 1
    )
    X_val_fold_mel_2048 = X_val_fold_mel_2048_s.reshape(
        X_val_fold_mel_2048.shape[0],
        X_val_fold_mel_2048.shape[1],
        X_val_fold_mel_2048.shape[2], 1
    )
    X_test_fold_mel_2048 = X_test_fold_mel_2048_s.reshape(
        X_test_fold_mel_2048.shape[0],
        X_test_fold_mel_2048.shape[1],
        X_test_fold_mel_2048.shape[2], 1
    )
    
    
    train_mask_t = np.isin(groups_tz, train_song_ids)
    val_mask_t = np.isin(groups_tz, val_song_ids)

    X_train_fold_t = X_train_t_aug_combined[train_mask_t]
    y_train_fold_t = y_train_t_aug_combined[train_mask_t]
    X_val_fold_t = X_train_t_aug_combined[val_mask_t]
    y_val_fold_t = y_train_t_aug_combined[val_mask_t]
    X_test_fold_t = X_test_t_aug_combined
    
    scaler_fold_t = StandardScaler()
    X_train_fold_t_s = scaler_fold_t.fit_transform(X_train_fold_t)
    X_val_fold_t_s = scaler_fold_t.transform(X_val_fold_t)
    X_test_fold_t_s = scaler_fold_t.transform(X_test_fold_t)
    
    X_train_fold_t = X_train_fold_t_s
    X_val_fold_t = X_val_fold_t_s
    X_test_fold_t = X_test_fold_t_s

    
    col_start = 0
    
    for i, name in enumerate(ensemble_models_name):
        print(f"Model {name} Fold {fold} training started.")
        if (name == "mel_2048_cnn_gru"):
            model = create_cnn_gru_model(X_train_fold_mel_2048.shape[1:])
            history, model = train_model(model, X_train_fold_mel_2048, y_train_fold_mel_2048, X_val_fold_mel_2048, y_val_fold_mel_2048, epochs=50, verbose=0)
            y_val_pred, accuracy, loss = evaluate_model_song_level(model, X_val_fold_mel_2048, y_val_fold_mel_2048, genres, "mel", visualize=False, return_predicted_probs = True)
            y_test_pred, _, _ = evaluate_model_song_level(model, X_test_fold_mel_2048, y_test_aug_mel_2048_combined, genres, "mel", visualize=False, return_predicted_probs = True)
        elif name == "mel_1024_cnn_bilstm":
            model = create_cnn_bilstm_model(X_train_fold_mel_1024.shape[1:])
            history, model = train_model(model, X_train_fold_mel_1024, y_train_fold_mel_1024, X_val_fold_mel_1024, y_val_fold_mel_1024, epochs=50, verbose=0)
            y_val_pred, accuracy, loss = evaluate_model_song_level(model, X_val_fold_mel_1024, y_val_fold_mel_1024, genres, "mel", visualize=False, return_predicted_probs = True)
            y_test_pred, _, _ = evaluate_model_song_level(model, X_test_fold_mel_1024, y_test_aug_mel_1024_combined, genres, "mel", visualize=False, return_predicted_probs = True)
        elif name == "mel_2048_cnn_lstm":
            model = create_cnn_lstm_model(X_train_fold_mel_2048.shape[1:])
            history, model = train_model(model, X_train_fold_mel_2048, y_train_fold_mel_2048, X_val_fold_mel_2048, y_val_fold_mel_2048, epochs=50, verbose=0)
            y_val_pred, accuracy, loss = evaluate_model_song_level(model, X_val_fold_mel_2048, y_val_fold_mel_2048, genres, "mel", visualize=False, return_predicted_probs = True)
            y_test_pred, _, _ = evaluate_model_song_level(model, X_test_fold_mel_2048, y_test_aug_mel_2048_combined, genres, "mel", visualize=False, return_predicted_probs = True)
        else:
            model = create_mlp_model(X_train_fold_t.shape[1:])
            history, model = train_model(model, X_train_fold_t, y_train_fold_t, X_val_fold_t, y_val_fold_t, epochs=50, batch_size=16, verbose=0)
            y_val_pred, accuracy, loss = evaluate_model_song_level(model, X_val_fold_t, y_val_fold_t, genres, "tzanetakis", visualize=False, return_predicted_probs = True)
            y_test_pred, _, _ = evaluate_model_song_level(model, X_test_fold_t, y_test_t_aug_combined, genres, "tzanetakis", visualize=False, return_predicted_probs = True)
        
        print(f"Model {name} Fold {fold} training completed.")
        oof_predictions[val_song_ids, col_start:col_start+n_classes] = y_val_pred
        test_predictions[:, col_start:col_start+n_classes, fold] = y_test_pred
        col_start += n_classes
        model_metrics[name]['acc'].append(accuracy)
        model_metrics[name]['loss'].append(loss)
        print(f"Fold {fold} Metrics for Model {name}")
        print(f"Accuracy: {model_metrics[name]['acc'][fold]}")
        print(f"Loss: {model_metrics[name]['loss'][fold]}")

test_predictions = np.mean(test_predictions, axis=2)

print("\n" + "="*40)
print("Cross-Validation Metrics Report")
print("="*40)
for name in ensemble_models_name:
    acc_mean = np.mean(model_metrics[name]['acc'])
    acc_std = np.std(model_metrics[name]['acc'])
    loss_mean = np.mean(model_metrics[name]['loss'])
    loss_std = np.std(model_metrics[name]['loss'])
    print(f"Model: {name}")
    print(f"  Accuracy: {acc_mean:.4f} (+/- {acc_std:.4f})")
    print(f"  Loss:     {loss_mean:.4f} (+/- {loss_std:.4f})")
    print("-" * 30)

In [ ]:
colnames = [f"{g}_{i}" for i in range(2) for g in genres]
oof_predictions_df = pd.DataFrame(oof_predictions, columns=colnames)
test_predictions_df = pd.DataFrame(test_predictions, columns=colnames)
oof_predictions_df.to_csv("oof_predictions_5_fold_2_models.csv")
test_predictions_df.to_csv("test_predictions_5_fold_2_models.csv")

In [ ]:
input_layer = Input(shape=oof_predictions.shape[1:])
x = input_layer
x = Dense(64, activation = 'relu')(x)
x = Dense(64, activation = 'relu')(x)
x = Dense(64, activation='relu')(x)

x = Dropout(0.3)(x)

output_layer = Dense(10, activation='softmax')(x)

meta_model = Model(inputs=input_layer, outputs=output_layer)

meta_model.compile(optimizer='adam',
                        loss='categorical_crossentropy',
                        metrics=['accuracy'])


def get_y_song(y_segments, num_of_segments):
    num_songs = len(y_segments) // num_of_segments
    total_valid_segments = num_songs * num_of_segments
        
    y_truncated = y_segments[:total_valid_segments]
    
    y_song_true = []
    
    for i in range(num_songs):
        start_idx = i * num_of_segments
        true_label = y_truncated[start_idx]
        y_song_true.append(true_label)
    y_song_true = np.array(y_song_true)
    return y_song_true

y_song_true = get_y_song(y_train_aug_mel_1024_combined, 19)
y_song_true_test = get_y_song(y_test_aug_mel_1024_combined, 19)

meta_model.fit(oof_predictions, y_song_true, epochs=50)

y_test_pred = meta_model.predict(test_predictions)

loss, acc = meta_model.evaluate(test_predictions, y_song_true_test)

print("Meta Model Test Evaluation")
print(f"Accuracy: {acc}")
print(f"Loss: {loss}")

In [ ]:
y_test1 = get_y_song(y_test_aug_mel_1024_combined, 19)
y_test2 = get_y_song(y_test_t_aug, 10)

In [ ]:
np.array_equal(y_test1, y_test2)

In [ ]:
y_train_song_aligned = y_train_aug_mel_1024_combined[::19]
np.array_equal(y_train_song_aligned, y_song_true)

In [ ]:
y_train_song_aligned = y_train_aug_mel_1024_combined[::19]  # Take first segment of each song

# Verify alignment
print(f"OOF predictions shape: {oof_predictions.shape}")
print(f"y_train_song_aligned shape: {y_train_song_aligned.shape}")
assert oof_predictions.shape[0] == y_train_song_aligned.shape[0], "Shape mismatch!"

# Check for any missing predictions (rows that are all zeros)
missing_predictions = (oof_predictions == 0).all(axis=1).sum()
print(f"Rows with missing predictions: {missing_predictions}")

# 2. Convert one-hot to class labels for Ridge
y_train_labels = np.argmax(y_train_song_aligned, axis=1)
y_test_labels = np.argmax(y_song_true_test, axis=1)

# 3. Split OOF predictions for validation
from sklearn.model_selection import train_test_split

oof_train, oof_val, y_train_meta, y_val_meta = train_test_split(
    oof_predictions, y_train_labels, 
    test_size=0.1, 
    stratify=y_train_labels, 
    random_state=42
)

# 4. Tune Ridge alpha with cross-validation on OOF predictions
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import RidgeClassifier

param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

ridge_meta = RidgeClassifier(random_state=42)
grid_search = GridSearchCV(
    ridge_meta, 
    param_grid, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1
)

print("\nTuning Ridge meta-model...")
grid_search.fit(oof_train, y_train_meta)

print(f"Best alpha: {grid_search.best_params_['alpha']}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

# 5. Train final meta-model with best alpha
meta_model_ridge = RidgeClassifier(alpha=grid_search.best_params_['alpha'], random_state=42)
meta_model_ridge.fit(oof_predictions, y_train_labels)

# 6. Evaluate on validation set
y_val_pred = meta_model_ridge.predict(oof_val)
val_acc = accuracy_score(y_val_meta, y_val_pred)
print(f"\nValidation Accuracy: {val_acc:.4f}")

# 7. Predict on test set
y_test_pred_ridge = meta_model_ridge.predict(test_predictions)
test_acc_ridge = accuracy_score(y_test_labels, y_test_pred_ridge)

print("\n" + "="*60)
print("Ridge Meta-Model Test Evaluation")
print("="*60)
print(f"Test Accuracy: {test_acc_ridge:.4f}")

# 8. Detailed evaluation
print("\nClassification Report:")
print(classification_report(y_test_labels, y_test_pred_ridge, target_names=genres))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test_labels, y_test_pred_ridge)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=genres, yticklabels=genres)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Ridge Meta-Model Confusion Matrix')
plt.show()

# 9. Compare with individual models
print("\n" + "="*60)
print("Comparison with Base Models")
print("="*60)
for name in ensemble_models_name:
    acc_mean = np.mean(model_metrics[name]['acc'])
    print(f"{name}: {acc_mean:.4f}")
print(f"Ridge Ensemble: {test_acc_ridge:.4f}")
print("="*60)

In [ ]:
# 1. Check OOF predictions distribution
print("="*60)
print("OOF PREDICTIONS DIAGNOSTIC")
print("="*60)

print(f"\nOOF predictions shape: {oof_predictions.shape}")
print(f"Min value: {oof_predictions.min()}")
print(f"Max value: {oof_predictions.max()}")
print(f"Mean value: {oof_predictions.mean()}")

# Check for rows with all zeros (songs that were never in validation)
zero_rows = (oof_predictions == 0).all(axis=1).sum()
print(f"\nRows with all zeros: {zero_rows}")

# Check distribution of predictions across classes
for i, name in enumerate(ensemble_models_name):
    col_start = i * 10
    col_end = col_start + 10
    model_preds = oof_predictions[:, col_start:col_end]
    pred_classes = np.argmax(model_preds, axis=1)
    print(f"\n{name} prediction distribution:")
    unique, counts = np.unique(pred_classes, return_counts=True)
    for cls, cnt in zip(unique, counts):
        print(f"  Class {cls} ({genres[cls]}): {cnt} songs")

# 2. Check if labels are aligned correctly
y_train_song_aligned = y_train_aug_mel_1024_combined[::19]
y_train_labels = np.argmax(y_train_song_aligned, axis=1)

print(f"\n\nTrue label distribution:")
unique, counts = np.unique(y_train_labels, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls} ({genres[cls]}): {cnt} songs")

# 3. Check test predictions
print(f"\n\nTest predictions shape: {test_predictions.shape}")
print(f"Min value: {test_predictions.min()}")
print(f"Max value: {test_predictions.max()}")

for i, name in enumerate(ensemble_models_name):
    col_start = i * 10
    col_end = col_start + 10
    model_test_preds = test_predictions[:, col_start:col_end]
    pred_classes = np.argmax(model_test_preds, axis=1)
    print(f"\n{name} TEST prediction distribution:")
    unique, counts = np.unique(pred_classes, return_counts=True)
    for cls, cnt in zip(unique, counts):
        print(f"  Class {cls} ({genres[cls]}): {cnt} songs")

# 4. Check a sample of predictions
print("\n\nSample of OOF predictions (first 3 songs, first model):")
print(oof_predictions[:3, :10])

# Add Dangdut Class

In [ ]:
all_files_path_d = []
all_genres_d = []

genres_d = genres + ['dangdut']

audio_path_d = '/kaggle/input/music-genre-classification-dataset/dangdut/thumbnails'

for i, genre in enumerate(genres):
    genre_path = audio_path + genre
    for _, dirname, files in os.walk(genre_path):
        for file in files:
            all_files_path_d.append(genre_path + "/" + file)
            all_genres_d.append(i)

for _, dirname, files in os.walk(audio_path_d):
    for file in files:
        all_files_path_d.append(audio_path_d + "/" + file)
        all_genres_d.append(10)
        

# Split Train Val Test 8:1:1
X_file_train_d, X_file_test_d, y_file_train_d, y_file_test_d = train_test_split(all_files_path_d, all_genres_d, test_size = 0.1, 
                                                                        stratify=all_genres_d, random_state=42)
X_file_train_d, X_file_val_d, y_file_train_d, y_file_val_d = train_test_split(X_file_train_d, y_file_train_d, test_size = (1/9), 
                                                                      stratify=y_file_train_d, random_state=42)

In [ ]:
print("🔃 Extracting Mel Features (n_fft = 1024) for X train...")
train_aug_mel_1024_d = extract_mel_features_from_file_path(
    X_file_train_d, y_file_train_d,
    augmentation=False, split_type="train",
    n_fft=1024, segment_overlap=0.5, new_class=True, num_classes=11
)
X_train_aug_mel_1024_d = train_aug_mel_1024_d['features']
y_train_aug_mel_1024_d = train_aug_mel_1024_d['labels']

print("🔃 Extracting Mel Features (n_fft = 2048) for X train...")
train_aug_mel_2048_d = extract_mel_features_from_file_path(
    X_file_train_d, y_file_train_d,
    augmentation=False, split_type="train",
    n_fft=2048, segment_overlap=0.5, new_class=True, num_classes=11
)
X_train_aug_mel_2048_d = train_aug_mel_2048_d['features']
y_train_aug_mel_2048_d = train_aug_mel_2048_d['labels']

In [ ]:
print("🔃 Extracting Tzanetakis Augmented Features for X train...")
train_t_aug_d = build_t_aug_feature_dataset(
    X_file_train_d, y_file_train_d,
    split_type="train", new_class=True, num_classes=11
)
X_train_t_aug_d = train_t_aug_d['features']
y_train_t_aug_d = train_t_aug_d['labels']

In [ ]:
print("🔃 Extracting Tzanetakis Matrix Augmented Features for X train...")
train_m_aug_d = build_matrix_aug_dataset(
    X_file_train_d, y_file_train_d,
    split_type="train", new_class=True, num_classes=11
)
X_train_m_aug_d = train_m_aug_d['features']
y_train_m_aug_d = train_m_aug_d['labels']


In [ ]:
print("🔃 Extracting Mel Features (n_fft = 1024) for X val...")
val_aug_mel_1024_d = extract_mel_features_from_file_path(
    X_file_val_d, y_file_val_d,
    augmentation=False, split_type="val",
    n_fft=1024, segment_overlap=0.5, new_class=True, num_classes=11
)
X_val_aug_mel_1024_d = val_aug_mel_1024_d['features']
y_val_aug_mel_1024_d = val_aug_mel_1024_d['labels']

print("🔃 Extracting Mel Features (n_fft = 2048) for X val...")
val_aug_mel_2048_d = extract_mel_features_from_file_path(
    X_file_val_d, y_file_val_d,
    augmentation=False, split_type="val",
    n_fft=2048, segment_overlap=0.5, new_class=True, num_classes=11
)
X_val_aug_mel_2048_d = val_aug_mel_2048_d['features']
y_val_aug_mel_2048_d = val_aug_mel_2048_d['labels']

print("🔃 Extracting Tzanetakis Augmented Features for X val...")
val_t_aug_d = build_t_aug_feature_dataset(
    X_file_val_d, y_file_val_d,
    split_type="val", new_class=True, num_classes=11
)
X_val_t_aug_d = val_t_aug_d['features']
y_val_t_aug_d = val_t_aug_d['labels']

print("🔃 Extracting Tzanetakis Matrix Augmented Features for X val...")
val_m_aug_d = build_matrix_aug_dataset(
    X_file_val_d, y_file_val_d,
    split_type="val", new_class=True, num_classes=11
)
X_val_m_aug_d = val_m_aug_d['features']
y_val_m_aug_d = val_m_aug_d['labels']


In [ ]:
print("🔃 Extracting Mel Features (n_fft = 1024) for X test...")
test_aug_mel_1024_d = extract_mel_features_from_file_path(
    X_file_test_d, y_file_test_d,
    augmentation=False, split_type="test",
    n_fft=1024, segment_overlap=0.5, new_class=True, num_classes=11
)
X_test_aug_mel_1024_d = test_aug_mel_1024_d['features']
y_test_aug_mel_1024_d = test_aug_mel_1024_d['labels']

print("🔃 Extracting Mel Features (n_fft = 2048) for X test...")
test_aug_mel_2048_d = extract_mel_features_from_file_path(
    X_file_test_d, y_file_test_d,
    augmentation=False, split_type="test",
    n_fft=2048, segment_overlap=0.5, new_class=True, num_classes=11
)
X_test_aug_mel_2048_d = test_aug_mel_2048_d['features']
y_test_aug_mel_2048_d = test_aug_mel_2048_d['labels']

print("🔃 Extracting Tzanetakis Augmented Features for X test...")
test_t_aug_d = build_t_aug_feature_dataset(
    X_file_test_d, y_file_test_d,
    split_type="test", new_class=True, num_classes=11
)
X_test_t_aug_d = test_t_aug_d['features']
y_test_t_aug_d = test_t_aug_d['labels']

print("🔃 Extracting Tzanetakis Matrix Augmented Features for X test...")
test_m_aug_d = build_matrix_aug_dataset(
    X_file_test_d, y_file_test_d,
    split_type="test", new_class=True, num_classes=11
)
X_test_m_aug_d = test_m_aug_d['features']
y_test_m_aug_d = test_m_aug_d['labels']

In [ ]:
# Normalize 1024
X_train_aug_mel_1024_reshaped_d = X_train_aug_mel_1024_d.reshape(X_train_aug_mel_1024_d.shape[0], -1)
X_test_aug_mel_1024_reshaped_d = X_test_aug_mel_1024_d.reshape(X_test_aug_mel_1024_d.shape[0], -1)
X_val_aug_mel_1024_reshaped_d = X_val_aug_mel_1024_d.reshape(X_val_aug_mel_1024_d.shape[0], -1)
scaler_aug_mel_1024_d = StandardScaler()
X_train_aug_mel_1024_scaled_d = scaler_aug_mel_1024_d.fit_transform(X_train_aug_mel_1024_reshaped_d)
X_test_aug_mel_1024_scaled_d = scaler_aug_mel_1024_d.transform(X_test_aug_mel_1024_reshaped_d)
X_val_aug_mel_1024_scaled_d = scaler_aug_mel_1024_d.transform(X_val_aug_mel_1024_reshaped_d)
X_train_aug_mel_1024_d = X_train_aug_mel_1024_scaled_d.reshape(X_train_aug_mel_1024_d.shape[0], X_train_aug_mel_1024_d.shape[1], X_train_aug_mel_1024_d.shape[2])
X_val_aug_mel_1024_d = X_val_aug_mel_1024_scaled_d.reshape(X_val_aug_mel_1024_d.shape[0], X_val_aug_mel_1024_d.shape[1], X_val_aug_mel_1024_d.shape[2])
X_test_aug_mel_1024_d = X_test_aug_mel_1024_scaled_d.reshape(X_test_aug_mel_1024_d.shape[0], X_test_aug_mel_1024_d.shape[1], X_test_aug_mel_1024_d.shape[2])

# Reshape for 2D-CNN 1024
X_train_aug_mel_1024_d = X_train_aug_mel_1024_d.reshape(X_train_aug_mel_1024_d.shape[0], X_train_aug_mel_1024_d.shape[1], X_train_aug_mel_1024_d.shape[2], 1)
X_val_aug_mel_1024_d = X_val_aug_mel_1024_d.reshape(X_val_aug_mel_1024_d.shape[0], X_val_aug_mel_1024_d.shape[1], X_val_aug_mel_1024_d.shape[2], 1)
X_test_aug_mel_1024_d = X_test_aug_mel_1024_d.reshape(X_test_aug_mel_1024_d.shape[0], X_test_aug_mel_1024_d.shape[1], X_test_aug_mel_1024_d.shape[2], 1)

print(f"Training 1024 shape: {X_train_aug_mel_1024_d.shape}")
print(f"Validation 1024 shape: {X_val_aug_mel_1024_d.shape}")
print(f"Test 1024 shape: {X_test_aug_mel_1024_d.shape}")

# Normalize 2048
X_train_aug_mel_2048_reshaped_d = X_train_aug_mel_2048_d.reshape(X_train_aug_mel_2048_d.shape[0], -1)
X_test_aug_mel_2048_reshaped_d = X_test_aug_mel_2048_d.reshape(X_test_aug_mel_2048_d.shape[0], -1)
X_val_aug_mel_2048_reshaped_d = X_val_aug_mel_2048_d.reshape(X_val_aug_mel_2048_d.shape[0], -1)
scaler_aug_mel_2048_d = StandardScaler()
X_train_aug_mel_2048_scaled_d = scaler_aug_mel_2048_d.fit_transform(X_train_aug_mel_2048_reshaped_d)
X_test_aug_mel_2048_scaled_d = scaler_aug_mel_2048_d.transform(X_test_aug_mel_2048_reshaped_d)
X_val_aug_mel_2048_scaled_d = scaler_aug_mel_2048_d.transform(X_val_aug_mel_2048_reshaped_d)
X_train_aug_mel_2048_d = X_train_aug_mel_2048_scaled_d.reshape(X_train_aug_mel_2048_d.shape[0], X_train_aug_mel_2048_d.shape[1], X_train_aug_mel_2048_d.shape[2])
X_val_aug_mel_2048_d = X_val_aug_mel_2048_scaled_d.reshape(X_val_aug_mel_2048_d.shape[0], X_val_aug_mel_2048_d.shape[1], X_val_aug_mel_2048_d.shape[2])
X_test_aug_mel_2048_d = X_test_aug_mel_2048_scaled_d.reshape(X_test_aug_mel_2048_d.shape[0], X_test_aug_mel_2048_d.shape[1], X_test_aug_mel_2048_d.shape[2])

# Reshape for 2D-CNN 2048
X_train_aug_mel_2048_d = X_train_aug_mel_2048_d.reshape(X_train_aug_mel_2048_d.shape[0], X_train_aug_mel_2048_d.shape[1], X_train_aug_mel_2048_d.shape[2], 1)
X_val_aug_mel_2048_d = X_val_aug_mel_2048_d.reshape(X_val_aug_mel_2048_d.shape[0], X_val_aug_mel_2048_d.shape[1], X_val_aug_mel_2048_d.shape[2], 1)
X_test_aug_mel_2048_d = X_test_aug_mel_2048_d.reshape(X_test_aug_mel_2048_d.shape[0], X_test_aug_mel_2048_d.shape[1], X_test_aug_mel_2048_d.shape[2], 1)

print(f"Training 2048 shape: {X_train_aug_mel_2048_d.shape}")
print(f"Validation 2048 shape: {X_val_aug_mel_2048_d.shape}")
print(f"Test 2048 shape: {X_test_aug_mel_2048_d.shape}")

# Prepare Augmented Data for 2D-CNN

# Normalize
scaler_t_aug_d = StandardScaler()
X_train_t_aug_scaled_d = scaler_t_aug_d.fit_transform(X_train_t_aug_d)
X_test_t_aug_scaled_d = scaler_t_aug_d.transform(X_test_t_aug_d)
X_val_t_aug_scaled_d = scaler_t_aug_d.transform(X_val_t_aug_d)

X_train_t_aug_d = X_train_t_aug_scaled_d
X_val_t_aug_d = X_val_t_aug_scaled_d
X_test_t_aug_d = X_test_t_aug_scaled_d

print(f"Training shape: {X_train_t_aug_d.shape}")
print(f"Validation shape: {X_val_t_aug_d.shape}")
print(f"Test shape: {X_test_t_aug_d.shape}")

# Prepare Augmented Data for 2D-CNN

# Normalize
X_train_m_aug_reshaped_d = X_train_m_aug_d.reshape(X_train_m_aug_d.shape[0], -1)
X_test_m_aug_reshaped_d = X_test_m_aug_d.reshape(X_test_m_aug_d.shape[0], -1)
X_val_m_aug_reshaped_d = X_val_m_aug_d.reshape(X_val_m_aug_d.shape[0], -1)
scaler_m_aug_d = StandardScaler()
X_train_m_aug_scaled_d = scaler_m_aug_d.fit_transform(X_train_m_aug_reshaped_d)
X_test_m_aug_scaled_d = scaler_m_aug_d.transform(X_test_m_aug_reshaped_d)
X_val_m_aug_scaled_d = scaler_m_aug_d.transform(X_val_m_aug_reshaped_d)
X_train_m_aug_d = X_train_m_aug_scaled_d.reshape(X_train_m_aug_d.shape[0], X_train_m_aug_d.shape[1], X_train_m_aug_d.shape[2])
X_val_m_aug_d = X_val_m_aug_scaled_d.reshape(X_val_m_aug_d.shape[0], X_val_m_aug_d.shape[1], X_val_m_aug_d.shape[2])
X_test_m_aug_d = X_test_m_aug_scaled_d.reshape(X_test_m_aug_d.shape[0], X_test_m_aug_d.shape[1], X_test_m_aug_d.shape[2])

# Reshape for 1D-CNN / RNN
X_train_m_aug_d = X_train_m_aug_d.reshape(X_train_m_aug_d.shape[0], X_train_m_aug_d.shape[2], X_train_m_aug_d.shape[1])
X_val_m_aug_d = X_val_m_aug_d.reshape(X_val_m_aug_d.shape[0], X_val_m_aug_d.shape[2], X_val_m_aug_d.shape[1])
X_test_m_aug_d = X_test_m_aug_d.reshape(X_test_m_aug_d.shape[0], X_test_m_aug_d.shape[2], X_test_m_aug_d.shape[1])

print(f"Training shape: {X_train_m_aug_d.shape}")
print(f"Validation shape: {X_val_m_aug_d.shape}")
print(f"Test shape: {X_test_m_aug_d.shape}")

In [ ]:
if(os.path.exists(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_lstm_model_d.h5')):
    mel_2048_cnn_lstm_model_d = load_model(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_lstm_model_d.h5')
    with open(MEL_2048_D_HISTORY_PATH + 'mel_2048_cnn_lstm_history_d.json', 'r') as f:
        mel_2048_cnn_lstm_history_d = json.load(f)
    print("Loaded existing mel_2048_cnn_lstm_model_d.h5 model and history.")

else:
    mel_2048_cnn_lstm_model_d = create_cnn_lstm_model(X_train_aug_mel_2048_d.shape[1:], num_classes=11)
    mel_2048_cnn_lstm_history_obj_d, mel_2048_cnn_lstm_model_d = train_model(
        mel_2048_cnn_lstm_model_d,
        X_train_aug_mel_2048_d, y_train_aug_mel_2048_d,
        X_val_aug_mel_2048_d, y_val_aug_mel_2048_d,
        epochs=50
    )
    mel_2048_cnn_lstm_history_d = mel_2048_cnn_lstm_history_obj_d.history

    save_model(mel_2048_cnn_lstm_model_d, 'model/mel_2048_cnn_lstm_model_d.h5')
    save_history(mel_2048_cnn_lstm_history_d, 'history/mel_2048_cnn_lstm_history_d.json')

plot_history(mel_2048_cnn_lstm_history_d)

print("Segment Level Evaluation")
mel_2048_cnn_lstm_test_acc_d, mel_2048_cnn_lstm_test_loss_d = evaluate_model(
    mel_2048_cnn_lstm_model_d, X_test_aug_mel_2048_d, y_test_aug_mel_2048_d, genres_d
)

print("Song Level Evaluation")
mel_2048_cnn_lstm_test_acc_song_d, mel_2048_cnn_lstm_test_loss_song_d = evaluate_model_song_level(
    mel_2048_cnn_lstm_model_d,
    X_test_aug_mel_2048_d,
    y_test_aug_mel_2048_d,
    genres_d,
    "mel"
)

In [ ]:
if(os.path.exists(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_gru_model_d.h5')):
    mel_2048_cnn_gru_model_d = load_model(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_gru_model_d.h5')
    with open(MEL_2048_D_HISTORY_PATH + 'mel_2048_cnn_gru_history_d.json', 'r') as f:
        mel_2048_cnn_gru_history_d = json.load(f)
    print("Loaded existing mel_2048_cnn_gru_model_d.h5 model and history.")

else:
    mel_2048_cnn_gru_model_d = create_cnn_gru_model(X_train_aug_mel_2048_d.shape[1:], num_classes=11)
    mel_2048_cnn_gru_history_obj_d, mel_2048_cnn_gru_model_d = train_model(
        mel_2048_cnn_gru_model_d,
        X_train_aug_mel_2048_d, y_train_aug_mel_2048_d,
        X_val_aug_mel_2048_d, y_val_aug_mel_2048_d,
        epochs=50
    )
    mel_2048_cnn_gru_history_d = mel_2048_cnn_gru_history_obj_d.history

    save_model(mel_2048_cnn_gru_model_d, 'model/mel_2048_cnn_gru_model_d.h5')
    save_history(mel_2048_cnn_gru_history_d, 'history/mel_2048_cnn_gru_history_d.json')

plot_history(mel_2048_cnn_gru_history_d)

print("Segment Level Evaluation")
mel_2048_cnn_gru_test_acc_d, mel_2048_cnn_gru_test_loss_d = evaluate_model(
    mel_2048_cnn_gru_model_d, X_test_aug_mel_2048_d, y_test_aug_mel_2048_d, genres_d
)

print("Song Level Evaluation")
mel_2048_cnn_gru_test_acc_song_d, mel_2048_cnn_gru_test_loss_song_d = evaluate_model_song_level(
    mel_2048_cnn_gru_model_d,
    X_test_aug_mel_2048_d,
    y_test_aug_mel_2048_d,
    genres_d,
    "mel"
)


In [ ]:
if(os.path.exists(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_bilstm_model_d.h5')):
    mel_2048_cnn_bilstm_model_d = load_model(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_bilstm_model_d.h5')
    with open(MEL_2048_D_HISTORY_PATH + 'mel_2048_cnn_bilstm_history_d.json', 'r') as f:
        mel_2048_cnn_bilstm_history_d = json.load(f)
    print("Loaded existing mel_2048_cnn_bilstm_model_d.h5 model and history.")

else:
    mel_2048_cnn_bilstm_model_d = create_cnn_bilstm_model(X_train_aug_mel_2048_d.shape[1:], num_classes=11)
    mel_2048_cnn_bilstm_history_obj_d, mel_2048_cnn_bilstm_model_d = train_model(
        mel_2048_cnn_bilstm_model_d,
        X_train_aug_mel_2048_d, y_train_aug_mel_2048_d,
        X_val_aug_mel_2048_d, y_val_aug_mel_2048_d,
        epochs=50
    )
    mel_2048_cnn_bilstm_history_d = mel_2048_cnn_bilstm_history_obj_d.history

    save_model(mel_2048_cnn_bilstm_model_d, 'model/mel_2048_cnn_bilstm_model_d.h5')
    save_history(mel_2048_cnn_bilstm_history_d, 'history/mel_2048_cnn_bilstm_history_d.json')

plot_history(mel_2048_cnn_bilstm_history_d)

print("Segment Level Evaluation")
mel_2048_cnn_bilstm_test_acc_d, mel_2048_cnn_bilstm_test_loss_d = evaluate_model(
    mel_2048_cnn_bilstm_model_d, X_test_aug_mel_2048_d, y_test_aug_mel_2048_d, genres_d
)

print("Song Level Evaluation")
mel_2048_cnn_bilstm_test_acc_song_d, mel_2048_cnn_bilstm_test_loss_song_d = evaluate_model_song_level(
    mel_2048_cnn_bilstm_model_d,
    X_test_aug_mel_2048_d,
    y_test_aug_mel_2048_d,
    genres_d,
    "mel"
)


In [ ]:
if(os.path.exists(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_bigru_model_d.h5')):
    mel_2048_cnn_bigru_model_d = load_model(MEL_2048_D_MODEL_PATH + 'mel_2048_cnn_bigru_model_d.h5')
    with open(MEL_2048_D_HISTORY_PATH + 'mel_2048_cnn_bigru_history_d.json', 'r') as f:
        mel_2048_cnn_bigru_history_d = json.load(f)
    print("Loaded existing mel_2048_cnn_bigru_model_d.h5 model and history.")

else:
    mel_2048_cnn_bigru_model_d = create_cnn_bigru_model(X_train_aug_mel_2048_d.shape[1:], num_classes=11)
    mel_2048_cnn_bigru_history_obj_d, mel_2048_cnn_bigru_model_d = train_model(
        mel_2048_cnn_bigru_model_d,
        X_train_aug_mel_2048_d, y_train_aug_mel_2048_d,
        X_val_aug_mel_2048_d, y_val_aug_mel_2048_d,
        epochs=50
    )
    mel_2048_cnn_bigru_history_d = mel_2048_cnn_bigru_history_obj_d.history

    save_model(mel_2048_cnn_bigru_model_d, 'model/mel_2048_cnn_bigru_model_d.h5')
    save_history(mel_2048_cnn_bigru_history_d, 'history/mel_2048_cnn_bigru_history_d.json')

plot_history(mel_2048_cnn_bigru_history_d)

print("Segment Level Evaluation")
mel_2048_cnn_bigru_test_acc_d, mel_2048_cnn_bigru_test_loss_d = evaluate_model(
    mel_2048_cnn_bigru_model_d, X_test_aug_mel_2048_d, y_test_aug_mel_2048_d, genres_d
)

print("Song Level Evaluation")
mel_2048_cnn_bigru_test_acc_song_d, mel_2048_cnn_bigru_test_loss_song_d = evaluate_model_song_level(
    mel_2048_cnn_bigru_model_d,
    X_test_aug_mel_2048_d,
    y_test_aug_mel_2048_d,
    genres_d,
    "mel"
)


In [ ]:
if(os.path.exists(TZANETAKIS_D_MODEL_PATH + 't_mlp_aug_model_d.h5')):
    t_mlp_aug_model_d = load_model(TZANETAKIS_D_MODEL_PATH + 't_mlp_aug_model_d.h5')
    with open(TZANETAKIS_D_HISTORY_PATH + 't_mlp_aug_history_d.json', 'r') as f:
        t_mlp_aug_model_history_d = json.load(f)
    print("Loaded existing t_mlp_aug_model_d.h5 model and history.")
else:
    t_mlp_aug_model_d = create_mlp_model(X_train_t_aug_d.shape[1:], num_classes=11)
    t_mlp_model_aug_history_obj_d, t_mlp_aug_model_d = train_model(
        t_mlp_aug_model_d,
        X_train_t_aug_d, y_train_t_aug_d,
        X_val_t_aug_d, y_val_t_aug_d,
        epochs=50,
        batch_size=16
    )
    t_mlp_aug_model_history_d = t_mlp_model_aug_history_obj_d.history

    save_model(t_mlp_aug_model_d, 'model/t_mlp_aug_model_d.h5')
    save_history(t_mlp_aug_model_history_d, 'history/t_mlp_aug_history_d.json')
    
plot_history(t_mlp_aug_model_history_d)

print("Segment Level Evaluation")
t_mlp_aug_model_test_acc_d, t_mlp_aug_model_test_loss_d = evaluate_model(
    t_mlp_aug_model_d, X_test_t_aug_d, y_test_t_aug_d, genres_d
)

print("Song Level Evaluation")
t_mlp_aug_model_test_acc_song_d, t_mlp_aug_model_test_loss_song_d = evaluate_model_song_level(
    t_mlp_aug_model_d,
    X_test_t_aug_d,
    y_test_t_aug_d,
    genres_d,
    "tzanetakis_vector"
)


In [ ]:
if(os.path.exists(TZANETAKIS_D_MODEL_PATH + 'm_lstm_aug_model_d.h5')):
    m_lstm_aug_model_d = load_model(TZANETAKIS_D_MODEL_PATH + 'm_lstm_aug_model_d.h5')
    with open(TZANETAKIS_D_HISTORY_PATH + 'm_lstm_aug_history_d.json', 'r') as f:
        m_lstm_aug_model_history_d = json.load(f)
    print("Loaded existing m_lstm_aug_model_d.h5 model and history.")
else:
    m_lstm_aug_model_d = create_lstm_model(X_train_m_aug_d.shape[1:], num_classes=11)
    m_lstm_model_aug_history_obj_d, m_lstm_aug_model_d = train_model(
        m_lstm_aug_model_d,
        X_train_m_aug_d, y_train_m_aug_d,
        X_val_m_aug_d, y_val_m_aug_d,
        epochs=50,
        batch_size=16
    )
    m_lstm_aug_model_history_d = m_lstm_model_aug_history_obj_d.history

    save_model(m_lstm_aug_model_d, 'model/m_lstm_aug_model_d.h5')
    save_history(m_lstm_aug_model_history_d, 'history/m_lstm_aug_history_d.json')
    
plot_history(m_lstm_aug_model_history_d)

print("Segment Level Evaluation")
m_lstm_aug_model_test_acc_d, m_lstm_aug_model_test_loss_d = evaluate_model(
    m_lstm_aug_model_d, X_test_m_aug_d, y_test_m_aug_d, genres_d
)

print("Song Level Evaluation")
m_lstm_aug_model_test_acc_song_d, m_lstm_aug_model_test_loss_song_d = evaluate_model_song_level(
    m_lstm_aug_model_d,
    X_test_m_aug_d,
    y_test_m_aug_d,
    genres_d,
    "tzanetakis_vector"
)
